# Face AI · Group 3 (video): the whole pipeline in one notebook

This notebook goes from the label table to every result, graph and curve, **one step at a time**.
Every step has the same four parts:

1. **What** we do, in plain words.
2. **Why** we do it.
3. **The code**, with comments on the important lines.
4. **What you should see**, and what it means.

| Tier | Model | Inputs |
|---|---|---|
| Baseline 1 | Always guess the training **median** | nothing |
| Baseline 2 | **Your CNN**, trained from scratch on 8 face crops | video |
| Improved | **Video specialists**: frozen face experts (DINOv2, CLIP) + skin-colour pulse signals, one tuned model per target | video |
| Advanced | **Multimodal specialists**: the video features plus finger PPG and ECG | video + contact sensors |

**How to run it**

* Open the notebook from the `Face_AI_2` folder and choose the kernel **`.venv` (Python 3.12)** of that folder.
* Click **Run All**. With every switch in Step 0.1 set to `False`, slow steps reuse their saved results and the whole
  notebook runs in under a minute.
* To redo a slow step, set its switch to `True`. PyTorch uses the NVIDIA GPU automatically when it can see it.
* Always run from the top (**Restart Kernel and Run All**). Running cells out of order can leave old variables behind.

## Contents

| Step | What happens |
|---|---|
| 0 | Settings, libraries, input files, names and colours |
| 1 | Look at the labels: what we must predict, and how much it varies |
| 2 | Split the **people** into folds (never split a person) |
| 3 | Baseline 1: the median guess, and the *skill* score |
| 4 | Turn the video into numbers: face embeddings and skin-colour pulse signals; proof that splitting by video cheats |
| 5 | Contact sensors: finger PPG and ECG features; the camera missed the exercise |
| 6 | Baseline 2: your CNN |
| 7 | The improved model: a specialist for every target, chosen by nested cross-validation |
| 8 | One shared network or separate models? |
| 9 | Evaluation: errors, confidence intervals, paired comparisons |
| 10 | All result graphs |
| 11 | Overfitting curves |
| 12 | Summary |
| 13 | Optional: update the presentation |

## Step 0 · Settings and tools

**What.** Choose which slow steps to recompute, load the libraries, point to the folders, and define the names and
colours used everywhere. **Why.** Everything later depends on these, so they live in one place.

In [ ]:
# ▶ Step 0.1 — Switches: False = reuse the saved result (fast); True = recompute that step
RECOMPUTE_DINO = False          # face embeddings with DINOv2              (about 3 min on CPU)
RECOMPUTE_CLIP = False          # face embeddings with CLIP                (about 4x DINOv2: 10-15 min on CPU)
RECOMPUTE_RPPG = False          # pulse features from skin colour          (seconds)
RECOMPUTE_PHYSIO = False        # finger-PPG and ECG features              (about 1 min; downloads 4.5 GB if missing)
RECOMPUTE_CNN = False           # your CNN baseline in 5 folds             (about 2.5 h on CPU)
CNN_QUICK_CHECK = False         # train the CNN for 1 epoch on 60 people, just to watch it run (about 1 min)
RECOMPUTE_SPECIALISTS = False   # nested CV for every specialist model     (about 13 min on CPU)
RECOMPUTE_MLP = False           # shared and single-task networks          (about 10 min on CPU)
RECOMPUTE_DIAGNOSTICS = False   # validation and learning curves           (about 1.5 min)
UPDATE_PRESENTATION = False     # rebuild presentation/index.html with this notebook's numbers
CPU_THREADS = 8                 # CPU threads NumPy and PyTorch may use

In [ ]:
# ▶ Step 0.2 — Libraries and folders
import os
os.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS))       # must be set before NumPy starts
import json, time, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 3)

PROJECT = Path.cwd()                               # the Face_AI_2 folder: open the notebook from there
OLD = PROJECT.parent / "Face_AI"                   # your other project folder (inputs are copied from it once)
DB_CSV = PROJECT / "data" / "db.csv"               # the dataset's label table
PHYSIO_DIR = PROJECT / "data" / "physio"           # finger-PPG and ECG files
FEATURES, RESULTS = PROJECT / "features", PROJECT / "results"
OOF, MET, FIG, LOGS = RESULTS / "oof", RESULTS / "metrics", RESULTS / "figures", RESULTS / "logs"
for d in (FEATURES, OOF, MET, FIG, LOGS):
    d.mkdir(parents=True, exist_ok=True)
assert DB_CSV.exists(), "Open this notebook from the Face_AI_2 folder (data/db.csv was not found)."
HF_REPO, HF_REVISION = "wengziheng/mcd_rppg", "91b967a5f2bab735d35f81722ff95e629358753b"
print("Project folder:", PROJECT)

In [ ]:
# ▶ Step 0.3 — Gather the input files into this project (copied once, so Face_AI_2 keeps working if other folders change)
CACHE = PROJECT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
SOURCES = {
    "faces.npz":    OLD / "baseline_test" / "cache" / "faces.npz",                    # 8 face crops per video
    "signals.npz":  OLD / "experiments" / "improved_cnn" / "cache" / "signals.npz",   # skin colour 30x per second
    "ppg_sync.npz": OLD / "experiments" / "improved_cnn" / "cache" / "ppg_sync.npz",  # finger pulse, same clock
}
for name, src in SOURCES.items():
    dst = CACHE / name
    if not dst.exists() and src.exists():
        shutil.copy2(src, dst)
        print(f"copied   {name:13s} {dst.stat().st_size / 1e6:6.0f} MB")
    elif dst.exists():
        print(f"ready    {name:13s} {dst.stat().st_size / 1e6:6.0f} MB")
    else:
        print(f"MISSING  {name}: expected at {src}")
FACES_NPZ, SIGNALS_NPZ, PPGSYNC_NPZ = CACHE / "faces.npz", CACHE / "signals.npz", CACHE / "ppg_sync.npz"
DINO_WEIGHTS = OLD / "experiments" / "improved_cnn" / "weights" / "dinov2-small"   # local DINOv2 copy (else downloaded)

In [ ]:
# ▶ Step 0.4 — The 10 targets, display names and units, and one fixed colour per model (used in every graph)
REG_TARGETS = ["saturation", "temperature", "glycated_hemoglobin", "cholesterol",
               "respiratory", "rigidity", "stress", "age", "bmi"]          # 9 numbers -> regression
ALL_TARGETS = REG_TARGETS + ["sex"]                                        # + 1 category -> classification
NAME = {"saturation": "SpO2 (%)", "temperature": "Temperature (°C)", "glycated_hemoglobin": "HbA1c (%)",
        "cholesterol": "Cholesterol (mmol/L)", "respiratory": "Breathing rate (/min)", "rigidity": "Arterial stiffness",
        "stress": "Stress (PSM-25)", "age": "Age (years)", "bmi": "BMI (kg/m²)", "sex": "Sex"}
SHORT = {"saturation": "SpO2", "temperature": "Temp", "glycated_hemoglobin": "HbA1c", "cholesterol": "Chol",
         "respiratory": "Breath", "rigidity": "Stiff", "stress": "Stress", "age": "Age", "bmi": "BMI", "sex": "Sex"}
UNIT = {"saturation": "%", "temperature": "°C", "glycated_hemoglobin": "%", "cholesterol": "mmol/L",
        "respiratory": "/min", "rigidity": "index", "stress": "score", "age": "years", "bmi": "kg/m²"}
LOG_TARGETS = ["age", "bmi"]          # right-skewed targets are modelled on a log scale, then turned back
N_FOLDS, N_REPEATS, SEED = 5, 3, 20261013

COLORS = {"constant": "#9b9a96", "cnn": "#eb6834", "video_specialists": "#2a78d6", "mtl_video_all": "#1baf7a",
          "multimodal_specialists": "#4a3aa7", "physio_only": "#eda100", "mtl_video_physio": "#e87ba4",
          "stl_video_all": "#008300", "state_constant": "#c9c8c3", "demo_oracle": "#6f6e69"}
LABEL = {"constant": "Baseline 1: median guess", "cnn": "Baseline 2: your CNN",
         "video_specialists": "Improved: video specialists", "mtl_video_all": "Shared network (video)",
         "stl_video_all": "Same network, one target at a time", "physio_only": "PPG + ECG only",
         "multimodal_specialists": "Advanced: video + PPG + ECG", "mtl_video_physio": "Shared network (multimodal)",
         "state_constant": "Reference: median per exercise state", "demo_oracle": "Reference: TRUE age/sex/BMI"}
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "legend.fontsize": 8, "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 100})
DIVERGING = LinearSegmentedColormap.from_list("skill", ["#c43a3a", "#e98d8d", "#f0efec", "#86b6ef", "#1c5cab"])


def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")      # every graph is also saved as a PNG

In [ ]:
# ▶ Step 0.5 — Small helpers used everywhere
def person_mae(abs_err, pids):
    """Average the errors within each person first, then across people: one vote per person."""
    return float(pd.Series(np.asarray(abs_err, float)).groupby(np.asarray(pids)).mean().mean())


def signed(v, d=2):
    s = f"{abs(v):.{d}f}"
    return s if float(s) == 0 else ("+" if v > 0 else "−") + s


def save_features(path, keys, X, names=None):
    extra = {"names": np.asarray(names, dtype=str)} if names is not None else {}      # plain text, never Python objects
    np.savez_compressed(path, keys=np.asarray(keys, dtype=str), features=np.asarray(X, np.float32), **extra)


def load_features(path, keys):
    """Rows of a saved feature file, in the same order as `keys`."""
    with np.load(path, allow_pickle=False) as z:
        pos = {str(k): i for i, k in enumerate(z["keys"])}
        return z["features"][[pos[str(k)] for k in keys]].astype(np.float64)


def save_oof(name, P, repeat):
    """Save out-of-fold predictions (one row per session) as results/oof/<name>_r<repeat>.csv."""
    out = df[["key", "patient_id", "step"]].copy()
    out["repeat"], out["fold"] = repeat, session_folds(repeat)
    for j, t in enumerate(ALL_TARGETS):
        out[f"pred_{t}"] = P[:, j]
    out.to_csv(OOF / f"{name}_r{repeat}.csv", index=False)


def load_oof(name, repeat=None):
    files = [f for f in sorted(OOF.glob(f"{name}_r*.csv")) if f.stem.rsplit("_r", 1)[0] == name
             and (repeat is None or f.stem.endswith(f"_r{repeat}"))]
    return pd.concat([pd.read_csv(f, dtype={"patient_id": str}) for f in files], ignore_index=True) if files else None

## Step 1 · Look at the labels

**What.** Read the dataset's label table and keep one row per **session** (a person, before or after 15 squats, filmed
by the frontal webcam). **Why.** Before modelling, we must know what each target is, how it was measured, and how much
it varies. A target that barely varies cannot be predicted much better than its typical value.

In [ ]:
# ▶ Step 1.1 — Read the label table and keep the frontal-webcam recordings
df = pd.read_csv(DB_CSV, dtype={"patient_id": str})
print("rows in db.csv:", len(df), "| cameras:", df["camera"].unique().tolist())
df = df[df["camera"] == "FullHDwebcam"].copy()                   # frontal camera only: 1 row per session
df["key"] = df["video"].map(lambda v: Path(v).stem)             # e.g. 1020_FullHDwebcam_after
df = df.sort_values("key").reset_index(drop=True)               # the same order as every feature file
df["sex_label"] = df["sex"].map({"F": 0.0, "M": 1.0})           # 0 = female, 1 = male
mixed = df.groupby("patient_id")["sex"].transform("nunique") > 1
df.loc[mixed, "sex_label"] = np.nan                             # 2 people have a different sex at their 2 sessions
df["after"] = (df["step"] == "after").astype(float)             # 1 = recorded after the squats
pids = df["patient_id"].to_numpy()
SESS = (df["patient_id"] + "_" + df["step"]).tolist()           # session id used by the PPG / ECG files
people = df.drop_duplicates("patient_id")
print(f"{len(df)} sessions from {df['patient_id'].nunique()} people; women/men = "
      f"{(people['sex'] == 'F').sum()}/{(people['sex'] == 'M').sum()}; sex hidden for {int(mixed.sum())} sessions")
df[["patient_id", "step", "age", "sex", "bmi", "saturation", "temperature", "respiratory", "rigidity",
    "glycated_hemoglobin", "cholesterol", "stress"]].head()

**What you should see.** 1,200 sessions from 600 people (399 women, 201 men). Each person appears twice: `before` and
`after` exercise.

In [ ]:
# ▶ Step 1.2 — How each target was measured, its spread, and whether it changes between a person's two sessions
MEASURED = {"saturation": "finger pulse oximeter", "temperature": "non-contact thermometer",
            "glycated_hemoglobin": "finger-prick test", "cholesterol": "finger-prick test",
            "respiratory": "counted by a doctor", "rigidity": "photoplethysmograph index",
            "stress": "PSM-25 questionnaire (item mean)", "age": "records", "bmi": "scale + height"}
rows = []
for t in REG_TARGETS:
    w = df.pivot(index="patient_id", columns="step", values=t)
    rows.append({"target": NAME[t], "measured with": MEASURED[t], "median": df[t].median(), "SD": df[t].std(),
                 "min": df[t].min(), "max": df[t].max(),
                 "% of people whose value changes": 100 * (w["before"] != w["after"]).mean(),
                 "mean before → after": f"{w['before'].mean():.2f} → {w['after'].mean():.2f}"})
pd.DataFrame(rows).set_index("target").round(2)

**What you should see.** Two kinds of targets:

* **Per session** (they change after exercise): breathing rate (+2.4 /min), arterial stiffness, SpO2, and sometimes temperature.
* **Per person** (identical in both sessions): age, sex, BMI, HbA1c, cholesterol and stress.

In [ ]:
# ▶ Step 1.3 — Graph: the distribution of every target, before vs after exercise
fig, axes = plt.subplots(3, 3, figsize=(11, 8))
for ax, t in zip(axes.flat, REG_TARGETS):
    lo, hi = np.percentile(df[t], [0.5, 99.5])
    if t in ("saturation", "respiratory"):
        bins = np.arange(np.floor(lo) - 0.5, hi + 1, 1)             # whole numbers: one bar per value
    elif t == "temperature":
        bins = np.arange(lo - 0.05, hi + 0.1, 0.1)                   # recorded in 0.1 °C steps
    else:
        bins = np.linspace(lo, hi, 26)
    for step, c in (("before", COLORS["video_specialists"]), ("after", COLORS["cnn"])):
        ax.hist(df.loc[df.step == step, t].clip(lo, hi), bins=bins, histtype="step", color=c, lw=2, label=f"{step} exercise")
    ax.axvline(df[t].median(), color=INK, lw=1)
    ax.set_title(NAME[t]); ax.set_ylabel("sessions")
axes[0, 0].legend(loc="upper left")
fig.suptitle("What we must predict: most targets sit in a narrow band around the median (black line)", y=1.0)
fig.tight_layout(); savefig(fig, "01_target_distributions"); plt.show()

**What it means.** SpO2 is 97–99 % for almost everyone and is recorded in whole percent. Temperature spreads over about
one thermometer step (0.1 °C). Breathing rate is counted in whole breaths. When the spread is about one tick of the
ruler, guessing the typical value is very hard to beat. Age is mostly 18–24, with a long tail of older people.

In [ ]:
# ▶ Step 1.4 — Does a label agree with itself? (same person, before vs after: "test–retest")
from scipy.stats import spearmanr
rt = []
for t in ["temperature", "saturation", "respiratory", "rigidity"]:
    w = df.pivot(index="patient_id", columns="step", values=t)
    rt.append({"target": NAME[t], "agreement ρ": spearmanr(w["before"], w["after"])[0],
               "identical values (%)": 100 * (w["before"] == w["after"]).mean()})
rt = pd.DataFrame(rt).set_index("target")
fig, ax = plt.subplots(figsize=(7.5, 2.6))
ax.barh(rt.index, rt["agreement ρ"], color=COLORS["video_specialists"], height=0.5)
for i, v in enumerate(rt["agreement ρ"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9, color=INK2)
ax.set_xlim(0, 1); ax.set_xlabel("agreement between a person's two sessions (Spearman ρ; 1 = identical ranking)")
ax.set_title("Arterial stiffness barely agrees with itself"); fig.tight_layout(); savefig(fig, "01b_test_retest"); plt.show()
rt.round(2)

**What it means.** Stiffness from the same person, minutes apart, agrees only weakly (ρ ≈ 0.28). Exercise explains part
of that, but it also shows measurement noise. No model can be more precise than the measurement it learns from.

## Step 2 · Split the people into folds

**What.** Give every **person** a fold number 0–4, three times with different shuffles. All sessions of a person get
the same fold.

**Why.** If the same face appears in training and in testing, the model can *recognise the person* and copy their
label, which makes scores look far better than they are (Step 4.5 proves it). With 600 people, one fixed test set of
90 people is also noisy. Cross-validation lets every person be tested once per shuffle by a model that never saw them.

**Nested cross-validation** has two loops:

* The **outer loop** scores: hold out one fold of people, train on the other four, predict the held-out people.
* The **inner loop** chooses: inside the four training folds, a second split picks the features, model and penalty.
  The held-out people are never used to choose anything.

In [ ]:
# ▶ Step 2.1 — Five folds of PEOPLE, stratified by sex x age band, repeated with 3 shuffles
from sklearn.model_selection import StratifiedGroupKFold
FOLDS_CSV = RESULTS / "folds.csv"
if FOLDS_CSV.exists():
    folds = pd.read_csv(FOLDS_CSV, dtype={"patient_id": str})
    print("loaded the saved folds:", FOLDS_CSV.name)
else:
    pp = people.set_index("patient_id")
    band = np.where(pp["age"] > 30, "old", np.where(pp["age"] > 22, "mid", "young"))
    strata = pp["sex"].astype(str) + "_" + band                       # keep sex and age mix similar in every fold
    rows = []
    for r in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (_, te) in enumerate(sgkf.split(np.zeros(len(pp)), strata.to_numpy(), groups=pp.index.to_numpy())):
            rows += [{"patient_id": p, "repeat": r, "fold": k} for p in pp.index[te]]
    folds = pd.DataFrame(rows)
    folds.to_csv(FOLDS_CSV, index=False)
    print("made new folds")


def session_folds(repeat):
    """Fold number (0-4) of every session row in df, for one shuffle."""
    f = folds[folds["repeat"] == repeat].set_index("patient_id")["fold"]
    return df["patient_id"].map(f).to_numpy()


for r in range(N_REPEATS):        # check: both sessions of every person share one fold
    assert pd.Series(session_folds(r)).groupby(pids).nunique().max() == 1
chk = people.merge(folds[folds.repeat == 0], on="patient_id")
chk.groupby("fold").agg(people=("patient_id", "size"), men=("sex", lambda s: (s == "M").sum()),
                        over_30=("age", lambda a: (a > 30).sum()), mean_age=("age", "mean")).round(1)

**What you should see.** 120 people per fold, about 40 men and 9 people over 30 in each. Every fold looks like the
whole cohort, so no fold is unusually easy or hard.

## Step 3 · Baseline 1: the median guess

**What.** For each target, predict the **training people's median** for everyone in the test fold. For sex, predict
the share of men.

**Why.** A model is only useful if it beats a guess that ignores the video. We score models with the **mean absolute
error (MAE)**, and the single guess with the lowest MAE is the median (the next graph shows it). We then rescale
every target to one footing:

$$\text{skill} = 1 - \frac{\text{MAE(model)}}{\text{MAE(median guess)}}$$

0 means no better than guessing, 1 means perfect, and below 0 means worse than guessing.

In [ ]:
# ▶ Step 3.1 — Graph: the average error of every possible single guess, for age
ages = people["age"].to_numpy(float)
guesses = np.linspace(17, 45, 300)
curve = np.array([np.mean(np.abs(ages - g)) for g in guesses])
med, mean = np.median(ages), ages.mean()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(guesses, curve, color=COLORS["video_specialists"])
for v, lab, c, dy in ((med, "median", INK, -16), (mean, "mean", COLORS["cnn"], 10)):
    e = np.mean(np.abs(ages - v))
    ax.scatter([v], [e], s=50, color=c, zorder=3, edgecolor=SURFACE)
    ax.annotate(f"{lab} {v:.1f} → error {e:.2f} years", (v, e), xytext=(10, dy), textcoords="offset points",
                fontsize=9, color=INK2)
ax.set_xlabel("one guess for everyone (years)"); ax.set_ylabel("average error (years)")
ax.set_title("The best single guess for the average absolute error is the median")
fig.tight_layout(); savefig(fig, "02_median_vs_mean"); plt.show()

In [ ]:
# ▶ Step 3.2 — Baseline 1 for every fold and shuffle, plus two reference "cheats" that help explain later results
Y_nat = np.column_stack([df[REG_TARGETS].to_numpy(float), df["sex_label"].to_numpy(float)])   # targets in real units
after = df["after"].to_numpy()
demo = np.column_stack([df["age"], np.log(df["bmi"]), df["sex"].eq("M").astype(float)])
for r in range(N_REPEATS):
    fold = session_folds(r)
    const = np.full((len(df), len(ALL_TARGETS)), np.nan)
    state, oracle = const.copy(), const.copy()
    for k in range(N_FOLDS):
        tr, te = fold != k, fold == k
        for j, t in enumerate(ALL_TARGETS):
            y = Y_nat[tr, j]
            ok = np.isfinite(y)
            if t == "sex":
                const[te, j] = state[te, j] = np.clip(np.nanmean(y), 0.01, 0.99)
                continue
            const[te, j] = np.median(y[ok])                                    # Baseline 1: the median guess
            for s in (0.0, 1.0):                                               # reference: median per exercise state
                state[te & (after == s), j] = np.median(y[ok & (after[tr] == s)])
            if t not in ("age", "bmi"):                                        # reference: TRUE age/BMI/sex -> target
                A = np.column_stack([np.ones(tr.sum()), demo[tr]])
                coef = np.linalg.lstsq(A[ok], y[ok], rcond=None)[0]
                shift = np.median(y[ok] - A[ok] @ coef)                        # aim at the median (MAE)
                oracle[te, j] = np.column_stack([np.ones(te.sum()), demo[te]]) @ coef + shift
    save_oof("constant", const, r)
    save_oof("state_constant", state, r)
    save_oof("demo_oracle", oracle, r)
c0 = load_oof("constant", 0)
pd.Series({NAME[t]: person_mae(np.abs(c0[f"pred_{t}"] - df[t]), pids) for t in REG_TARGETS},
          name="MAE of the median guess").round(3).to_frame()

**What you should see.** The error of the median guess in each target's own unit. For example, age is about 4.1 years
off and BMI about 3.2 kg/m². Every later model is compared with these numbers.

The two **references** are not deployable: one knows whether the person just exercised, the other knows their true
age, sex and BMI. They help explain later results.

## Step 4 · Turn the video into numbers

**What.** We make three kinds of numbers from each video:

1. **Face crops:** 8 frames per video, cut around the face (used directly by your CNN in Step 6).
2. **Frozen face embeddings:** DINOv2 and CLIP have already learned from hundreds of millions of images. They turn each
   crop into a list of numbers that describe the face. We never change their weights: they stay **frozen**.
3. **Skin-colour pulse signals (rPPG):** every heartbeat changes skin colour very slightly. The colour of the forehead
   and cheeks over time gives heart rate, its variability and breathing cues.

**Why.** 600 people are far too few to learn what a face is from scratch. Borrowing a pretrained expert, then training
only a small model on top, is called **transfer learning**.

In [ ]:
# ▶ Step 4.1 — The face crops: check their size without loading them all
with zipfile.ZipFile(FACES_NPZ) as zf, zf.open("frames.npy") as f:
    version = np.lib.format.read_magic(f)
    shape, _, dtype = (np.lib.format.read_array_header_1_0(f) if version == (1, 0)
                       else np.lib.format.read_array_header_2_0(f))
print("face crops:", shape, dtype, " ->  (videos, frames per video, height, width, RGB)")
with np.load(FACES_NPZ) as z:
    assert (z["keys"] == df["key"].to_numpy()).all(), "face crops are not in the label-table order"

In [ ]:
# ▶ Step 4.2 — Frozen face expert #1: DINOv2 (a vision transformer trained on 142 million images)
import torch
torch.set_num_threads(CPU_THREADS)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch is using:", DEVICE)


def embed_faces(model_name, out_path, kind):
    """Turn every face crop into numbers with a frozen encoder, then average the 8 crops of each video."""
    from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModel
    if kind == "clip":
        proc, model = CLIPImageProcessor.from_pretrained(model_name), CLIPVisionModel.from_pretrained(model_name)
    else:
        proc, model = AutoImageProcessor.from_pretrained(model_name), AutoModel.from_pretrained(model_name)
    model = model.eval().to(DEVICE)
    with np.load(FACES_NPZ) as z:
        keys, frames = z["keys"], z["frames"]                          # (1200, 8, 112, 112, 3)
    n, t = frames.shape[:2]
    feats, t0 = None, time.time()
    with torch.inference_mode():                                       # no training: just a forward pass
        for i in range(0, n, 16):                                      # 16 videos = 128 crops per batch
            crops = list(frames[i:i + 16].reshape(-1, *frames.shape[2:]))
            px = proc(images=crops, return_tensors="pt")["pixel_values"].to(DEVICE)
            out = model(pixel_values=px)
            vec = out.pooler_output if kind == "clip" else out.last_hidden_state[:, 0]   # one vector per crop
            vec = vec.view(-1, t, vec.shape[-1]).mean(1).cpu().numpy()                    # average the 8 crops
            if feats is None:
                feats = np.zeros((n, vec.shape[1]), np.float32)
            feats[i:i + 16] = vec
            if i % 320 == 0:
                print(f"  {min(i + 16, n)}/{n} videos  {time.time() - t0:.0f}s")
    save_features(out_path, keys, feats)
    print(f"saved {out_path.name} {feats.shape} in {time.time() - t0:.0f}s")


DINO_FILE = FEATURES / "dino_vits14.npz"
if RECOMPUTE_DINO or not DINO_FILE.exists():
    embed_faces(str(DINO_WEIGHTS) if DINO_WEIGHTS.exists() else "facebook/dinov2-small", DINO_FILE, "dino")
X_dino = load_features(DINO_FILE, df["key"])
print("DINOv2 features:", X_dino.shape, "-> 384 numbers per video")

In [ ]:
# ▶ Step 4.3 — Frozen face expert #2: CLIP ViT-B/16 (trained on 400 million image–caption pairs)
CLIP_FILE = FEATURES / "clip_vitb16.npz"
if RECOMPUTE_CLIP or not CLIP_FILE.exists():
    embed_faces("openai/clip-vit-base-patch16", CLIP_FILE, "clip")
X_clip = load_features(CLIP_FILE, df["key"])
print("CLIP features:", X_clip.shape, "-> 768 numbers per video")

### Step 4.4 · Skin colour → pulse signals (rPPG)

**What.** For each video we have the average red, green and blue of three skin patches (forehead, right cheek, left
cheek) 30 times per second for the first minute, plus 5 face landmarks. From those traces we compute:

* **Pulse rate** with three classic methods: *green* (green brightness), *CHROM* (two colour differences) and *POS*
  (a projection that cancels skin tone). That gives 3 patches × 3 methods, plus a quality score for each.
* **Breathing** from slow colour changes and from small up-and-down head movements.
* **Heart-rate variability** from the beat-to-beat intervals.
* **Skin tone** (mean colours and colour ratios) and signal quality.

**Why.** These are the only video features that can see the body *working*: the face embeddings only see appearance.

In [ ]:
# ▶ Step 4.4a — Signal-processing helpers
from scipy.signal import butter, filtfilt, welch, find_peaks
from numpy.lib.stride_tricks import sliding_window_view
FS = 30.0                                       # samples per second of the colour traces
HR_BAND, RESP_BAND = (0.7, 3.5), (0.13, 0.6)    # 42-210 beats/min and 8-36 breaths/min


def bandpass(x, lo, hi, fs=FS, order=3):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], "band")
    return filtfilt(b, a, x, axis=0)


def fill_gaps(x, valid):
    """Replace invalid samples with straight lines between their valid neighbours."""
    x = np.array(x, float)
    bad = ~np.asarray(valid, bool) | ~np.isfinite(x)
    if bad.all():
        return np.full_like(x, np.nan)
    idx = np.arange(len(x))
    x[bad] = np.interp(idx[bad], idx[~bad], x[~bad])
    return x


def green(rgb):
    return rgb[:, 1] / rgb[:, 1].mean() - 1


def chrom(rgb):
    """CHROM (de Haan & Jeanne, 2013): two colour-difference signals, mixed to cancel movement."""
    n = rgb / rgb.mean(0)
    X = bandpass(3 * n[:, 0] - 2 * n[:, 1], *HR_BAND)
    Y = bandpass(1.5 * n[:, 0] + n[:, 1] - 1.5 * n[:, 2], *HR_BAND)
    return X - (X.std() / (Y.std() + 1e-9)) * Y


def pos(rgb, win=1.6):
    """POS (Wang et al., 2017): in each 1.6 s window, project the normalised colours onto a plane that removes skin tone."""
    w = int(win * FS)
    W = sliding_window_view(rgb, (w, 3))[:, 0]                       # (windows, w, 3)
    Cn = W / W.mean(axis=1, keepdims=True)
    S1 = Cn[..., 1] - Cn[..., 2]
    S2 = -2 * Cn[..., 0] + Cn[..., 1] + Cn[..., 2]
    h = S1 + (S1.std(axis=1, keepdims=True) / (S2.std(axis=1, keepdims=True) + 1e-9)) * S2
    h = h - h.mean(axis=1, keepdims=True)
    out, cnt = np.zeros(len(rgb)), np.zeros(len(rgb))
    for i in range(w):                                               # overlap-add the windows
        out[i:i + len(h)] += h[:, i]
        cnt[i:i + len(h)] += 1
    return out / np.maximum(cnt, 1)


def spectrum_peak(x, band, fs=FS):
    """Strongest frequency in a band (Hz) and the share of in-band power within ±0.1 Hz of it (a quality score)."""
    x = np.asarray(x, float)
    if not np.isfinite(x).all() or x.std() == 0:
        return np.nan, np.nan
    f, p = welch(x - x.mean(), fs, nperseg=min(len(x), 512), nfft=4096)
    m = (f >= band[0]) & (f <= band[1])
    if m.sum() < 3 or p[m].sum() == 0:
        return np.nan, np.nan
    fpk = f[m][np.argmax(p[m])]
    return float(fpk), float(p[m][np.abs(f[m] - fpk) <= 0.1].sum() / p[m].sum())


with np.load(SIGNALS_NPZ) as z:
    assert (z["keys"] == df["key"].to_numpy()).all()
    RGB, VALID, LMK = z["rgb"], z["valid"], z["landmarks"]          # (1200, 1800, 9), (1200, 1800), (1200, 1800, 10)
    CHANNELS = list(z["channels"])
PPGS = np.load(PPGSYNC_NPZ)["ppg"] if PPGSYNC_NPZ.exists() else None  # finger pulse on the same 30 Hz clock
print("colour traces:", RGB.shape, "| channels:", CHANNELS[:3], "...")

In [ ]:
# ▶ Step 4.4b — Graph: one recording, from skin colour to a pulse wave, checked against the finger pulse sensor
i = int(np.where(df["key"] == "1020_FullHDwebcam_before")[0][0])    # any session works; this one is clean
t = np.arange(RGB.shape[1]) / FS
fore = np.column_stack([fill_gaps(RGB[i, :, c], VALID[i]) for c in range(3)])     # forehead R, G, B
cam = bandpass(pos(fore), *HR_BAND)
fig, axes = plt.subplots(3, 1, figsize=(10, 7.4))
axes[0].plot(t, fore[:, 1], color=COLORS["mtl_video_all"], lw=1)
axes[0].set_title("1. Forehead green brightness over one minute: the pulse hides in tiny wiggles")
axes[0].set_xlabel("seconds"); axes[0].set_ylabel("green level")
m10 = t < 10
axes[1].plot(t[m10], cam[m10] / cam[m10].std(), color=COLORS["video_specialists"], lw=1.6, label="camera pulse (POS)")
if PPGS is not None:
    finger = bandpass(fill_gaps(PPGS[i], np.isfinite(PPGS[i])), *HR_BAND)
    finger *= np.sign(np.corrcoef(cam, finger)[0, 1]) or 1           # the sensor's sign convention is arbitrary
    axes[1].plot(t[m10], finger[m10] / finger[m10].std(), color=COLORS["cnn"], lw=1.2, alpha=0.85, label="finger pulse sensor")
axes[1].set_title("2. After POS and a 0.7-3.5 Hz filter, a pulse wave appears (first 10 s)")
axes[1].set_xlabel("seconds"); axes[1].legend(loc="upper right")
for sig, c, lab in ((cam, COLORS["video_specialists"], "camera"), (finger if PPGS is not None else None, COLORS["cnn"], "finger")):
    if sig is None:
        continue
    f, p = welch(sig - sig.mean(), FS, nperseg=512, nfft=4096)
    m = (f > 0.5) & (f < 3.5)
    axes[2].plot(60 * f[m], p[m] / p[m].max(), color=c, lw=1.6, label=f"{lab}: peak {60 * f[m][np.argmax(p[m])]:.0f} bpm")
axes[2].set_title("3. The strongest frequency is the heart rate"); axes[2].set_xlabel("beats per minute")
axes[2].set_ylabel("relative power"); axes[2].legend()
fig.tight_layout(); savefig(fig, "03_rppg_example"); plt.show()

In [ ]:
# ▶ Step 4.4c — Pulse and breathing features for all 1,200 recordings
from joblib import Parallel, delayed
REGIONS = ["forehead", "right cheek", "left cheek"]
METHODS = {"green": green, "chrom": chrom, "pos": pos}


def rppg_features(rgb_all, valid, lmk):
    """Label-free numbers from one recording's colour traces and landmarks."""
    f = {}
    hrs, snrs = [], []
    for r, reg in enumerate(REGIONS):
        rgb = np.column_stack([fill_gaps(rgb_all[:, 3 * r + c], valid) for c in range(3)])
        usable = np.isfinite(rgb).all() and (rgb > 0).all()
        for mname, fn in METHODS.items():
            hz, q = spectrum_peak(bandpass(fn(rgb), *HR_BAND), HR_BAND) if usable else (np.nan, np.nan)
            f[f"{reg} {mname} pulse bpm"], f[f"{reg} {mname} pulse quality"] = 60 * hz, q
            hrs.append(60 * hz); snrs.append(q)
        hz, q = spectrum_peak(bandpass(green(rgb), 0.08, 0.7, order=2), RESP_BAND) if usable else (np.nan, np.nan)
        f[f"{reg} breathing bpm (colour)"], f[f"{reg} breathing strength (colour)"] = 60 * hz, q
        mean = rgb.mean(0) if usable else np.full(3, np.nan)
        f[f"{reg} mean R"], f[f"{reg} mean G"], f[f"{reg} mean B"] = mean
        f[f"{reg} R/G"], f[f"{reg} B/G"] = mean[0] / mean[1], mean[2] / mean[1]
        f[f"{reg} colour variability"] = float(np.mean(rgb.std(0) / rgb.mean(0))) if usable else np.nan
    hrs, snrs = np.array(hrs, float), np.array(snrs, float)
    good = np.isfinite(hrs) & np.isfinite(snrs)
    f["pulse bpm (median of 9)"] = np.median(hrs[good]) if good.any() else np.nan
    f["pulse bpm spread"] = np.std(hrs[good]) if good.any() else np.nan
    f["pulse bpm (best quality)"] = hrs[good][np.argmax(snrs[good])] if good.any() else np.nan
    f["pulse quality (best)"] = snrs[good].max() if good.any() else np.nan
    lm = np.column_stack([fill_gaps(lmk[:, c], valid) for c in range(lmk.shape[1])])
    for axis, sig in (("vertical", lm[:, 1::2].mean(1)), ("sideways", lm[:, 0::2].mean(1))):
        hz, q = spectrum_peak(bandpass(sig - sig.mean(), 0.08, 0.7, order=2), RESP_BAND)
        f[f"breathing bpm (head {axis})"], f[f"breathing strength (head {axis})"] = 60 * hz, q
        f[f"head movement {axis} (px)"] = float(np.std(sig))
    ibi = np.array([])
    if good.any():                                                   # beat-to-beat intervals from the best patch
        r = int(np.argmax(np.where(good, snrs, -1)) // 3)
        rgb = np.column_stack([fill_gaps(rgb_all[:, 3 * r + c], valid) for c in range(3)])
        pk, _ = find_peaks(bandpass(pos(rgb), *HR_BAND), distance=max(int(FS * 0.5 * 60 / f["pulse bpm (best quality)"]), 6))
        ibi = np.diff(pk) / FS
        ibi = ibi[(ibi > 0.3) & (ibi < 2.0)]
    f["beat interval SD (ms)"] = 1000 * ibi.std() if len(ibi) > 5 else np.nan
    f["beat interval RMSSD (ms)"] = 1000 * np.sqrt(np.mean(np.diff(ibi) ** 2)) if len(ibi) > 5 else np.nan
    f["valid fraction"] = float(np.mean(valid))
    return f


RPPG_FILE = FEATURES / "rppg_v2.npz"
if RECOMPUTE_RPPG or not RPPG_FILE.exists():
    t0 = time.time()
    rows = Parallel(n_jobs=CPU_THREADS)(delayed(rppg_features)(RGB[k], VALID[k], LMK[k]) for k in range(len(df)))
    rppg_names = list(rows[0])
    save_features(RPPG_FILE, df["key"], [[r[n] for n in rppg_names] for r in rows], rppg_names)
    print(f"computed in {time.time() - t0:.0f}s")
with np.load(RPPG_FILE) as z:
    rppg_names = list(z["names"])
X_rppg = load_features(RPPG_FILE, df["key"])
rppg = pd.DataFrame(X_rppg, columns=rppg_names)
print("rPPG features:", X_rppg.shape)
rppg[["pulse bpm (best quality)", "pulse quality (best)", "breathing bpm (head vertical)", "beat interval SD (ms)"]].describe().round(2)

In [ ]:
# ▶ Step 4.4d — Graph: does the camera pulse match the finger pulse sensor? (same minute, all 1,200 recordings)
if PPGS is not None:
    finger_hr = np.array([60 * spectrum_peak(bandpass(fill_gaps(PPGS[k], np.isfinite(PPGS[k])), *HR_BAND), HR_BAND)[0]
                          for k in range(len(df))])
    cam_hr = rppg["pulse bpm (best quality)"].to_numpy()
    ok = np.isfinite(finger_hr) & np.isfinite(cam_hr)
    err = np.abs(cam_hr[ok] - finger_hr[ok])
    fig, ax = plt.subplots(figsize=(5.4, 5))
    ax.scatter(finger_hr[ok], cam_hr[ok], s=10, alpha=0.5, color=COLORS["video_specialists"], edgecolor="none")
    ax.plot([40, 160], [40, 160], color=INK, lw=1)
    ax.set_xlim(40, 160); ax.set_ylim(40, 160)
    ax.set_xlabel("finger sensor heart rate (bpm)"); ax.set_ylabel("camera heart rate (bpm)")
    ax.set_title(f"Within 5 bpm for {100 * np.mean(err <= 5):.0f}% of recordings")
    fig.tight_layout(); savefig(fig, "04_camera_vs_finger_pulse"); plt.show()
    print(f"median difference {np.median(err):.1f} bpm; within 10 bpm: {100 * np.mean(err <= 10):.0f}%")

**What you should see.** Points on the diagonal are recordings where the camera and the finger agree. The two flat
bands, around 150 bpm and around 50 bpm, are not heart rates: in those videos the camera locked onto a steady
artefact (light flicker or head movement) instead of the pulse. That is why every estimate also carries a quality
score, and why the models receive the quality numbers too.

In [ ]:
# ▶ Step 4.5 — Proof: splitting by VIDEO lets the model cheat. Same model and data, only the split changes.
from sklearn.model_selection import KFold
X_face = np.column_stack([X_dino, X_clip])
leaky = np.zeros(len(df), int)
for k, (_, te) in enumerate(KFold(5, shuffle=True, random_state=0).split(X_face)):
    leaky[te] = k                                            # random videos: a person's 2 sessions can be split
LEAK_T = ["cholesterol", "glycated_hemoglobin", "stress", "age", "bmi"]
YL = np.column_stack([df[t].to_numpy(float) for t in LEAK_T])


def leak_skill(fold):
    from sklearn.kernel_ridge import KernelRidge
    from sklearn.preprocessing import StandardScaler
    P = np.zeros_like(YL)
    for k in range(5):
        tr, te = fold != k, fold == k
        sc = StandardScaler().fit(X_face[tr])
        Ztr, Zte = sc.transform(X_face[tr]) / np.sqrt(X_face.shape[1]), sc.transform(X_face[te]) / np.sqrt(X_face.shape[1])
        model = KernelRidge(alpha=0.1, kernel="rbf", gamma=0.5).fit(Ztr, YL[tr] - YL[tr].mean(0))
        P[te] = model.predict(Zte) + YL[tr].mean(0)
    out = {}
    for j, t in enumerate(LEAK_T):
        const = np.zeros(len(df))
        for k in range(5):
            const[fold == k] = np.median(YL[fold != k, j])
        out[NAME[t]] = 1 - person_mae(np.abs(P[:, j] - YL[:, j]), pids) / person_mae(np.abs(const - YL[:, j]), pids)
    return out


leak = pd.DataFrame({"split by video (cheating)": leak_skill(leaky), "split by person (honest)": leak_skill(session_folds(0))})
twin = np.mean([(leaky[pids == p] != leaky[i]).any() for i, p in enumerate(pids)])
fig, ax = plt.subplots(figsize=(8, 3.6))
yy = np.arange(len(leak))
ax.barh(yy + 0.18, leak.iloc[:, 0], height=0.34, color=COLORS["cnn"], label=leak.columns[0])
ax.barh(yy - 0.18, leak.iloc[:, 1], height=0.34, color=COLORS["video_specialists"], label=leak.columns[1])
ax.set_yticks(yy, leak.index); ax.axvline(0, color=INK, lw=1); ax.invert_yaxis()
ax.set_xlabel("skill vs the median guess"); ax.legend(loc="lower right")
ax.set_title(f"Split by video and {100 * twin:.0f}% of test videos have the same person's other video in training")
fig.tight_layout(); savefig(fig, "05_leakage_demo"); plt.show()
leak.round(3)

**What it means.** Split by video, the model looks as if it can read cholesterol, HbA1c and stress from a face. Split
by person, that skill disappears. The model was recognising faces and copying each person's label. **Every number in
this notebook is split by person.**

## Step 5 · Contact sensors: finger PPG and ECG

**What.** Each session also has a **finger PPG** (pulse wave, 100 Hz, about 180 s) and an **ECG** (heart's electrical
signal, 500 Hz, 30 s; leads V2–V6 are empty). We turn them into heart rate, heart-rate variability, pulse-wave shape and
breathing features. **Why.** These signals are cleaner than the camera's, so they show how much information exists in
the body's signals at all. A phone camera does not have them, so this tier is an **upper bound**, not a deployable model.

**Careful:** arterial stiffness was measured with a photoplethysmograph. A model that uses the finger PPG may partly
recompute that label from the same kind of signal.

In [ ]:
# ▶ Step 5.1 — Make sure the PPG and ECG files are here (downloads about 4.5 GB only if they are missing)
present = {s: len(list((PHYSIO_DIR / s).glob("*"))) for s in ("ppg", "ecg")}
print("files present:", present)
if min(present.values()) < 1200:
    from huggingface_hub import snapshot_download
    snapshot_download(HF_REPO, repo_type="dataset", revision=HF_REVISION, token=True, local_dir=PHYSIO_DIR,
                      allow_patterns=["ppg/*", "ecg/*"], max_workers=8)

In [ ]:
# ▶ Step 5.2 — Feature extraction for one PPG file and one ECG file
from scipy.interpolate import interp1d
from scipy.stats import skew
SENSOR_RESP = (0.13, 0.6)


def spectral_peak_fs(x, fs, band, nperseg=None):
    x = np.asarray(x, float) - np.nanmean(x)
    if len(x) < 16 or not np.isfinite(x).all() or np.std(x) == 0:
        return np.nan, np.nan
    f, p = welch(x, fs, nperseg=min(len(x), nperseg or len(x)))
    m = (f >= band[0]) & (f <= band[1])
    if m.sum() < 3:
        return np.nan, np.nan
    i = np.argmax(p[m])
    return float(f[m][i]), float(p[m][max(i - 1, 0):i + 2].sum() / p[m].sum())


def band_centroid(x, fs, band=(0.2, 0.45), nperseg=None):
    x = np.asarray(x, float)
    if len(x) < 16 or not np.isfinite(x).all() or np.std(x) == 0:
        return np.nan
    f, p = welch(x - x.mean(), fs, nperseg=min(len(x), nperseg or len(x)))
    m = (f >= band[0]) & (f <= band[1])
    return float((f[m] * p[m]).sum() / p[m].sum()) if p[m].sum() > 0 else np.nan


def resp_from_series(t, v, fs_out=4.0):
    """Breathing frequency from a beat-by-beat series (amplitudes or intervals) resampled at 4 Hz."""
    ok = np.isfinite(v)
    t, v = np.asarray(t)[ok], np.asarray(v)[ok]
    if len(t) < 10:
        return np.nan, np.nan, np.nan
    grid = np.arange(t[0], t[-1], 1 / fs_out)
    y = bandpass(interp1d(t, v)(grid), 0.1, 0.8, fs_out, order=2)
    f, frac = spectral_peak_fs(y, fs_out, SENSOR_RESP, nperseg=int(64 * fs_out))
    return f, frac, band_centroid(y, fs_out, nperseg=int(64 * fs_out))


def hrv(ibi_s):
    ibi = np.asarray(ibi_s) * 1000.0
    d = np.diff(ibi)
    return {"sdnn_ms": float(np.std(ibi)), "rmssd_ms": float(np.sqrt(np.mean(d ** 2))) if len(d) else np.nan,
            "pnn50": float(np.mean(np.abs(d) > 50)) if len(d) else np.nan, "ibi_cv": float(np.std(ibi) / np.mean(ibi))}


def clean_ibi(ibi):
    ok = (ibi > 0.3) & (ibi < 2.0)
    med = np.median(ibi[ok]) if ok.any() else np.nan
    return ok & (np.abs(ibi - med) < 0.3 * med)


def ppg_features(path):
    fs = 100.0
    raw = pd.read_csv(path, sep=r"\s+", header=None, usecols=[0]).iloc[:, 0].to_numpy(float)[200:-200]
    clipped = (raw <= 0) | (raw >= 255)
    x = -raw                                                         # the device's output is upside down
    xf = bandpass(x, 0.5, 8.0, fs)
    if skew(np.diff(xf)) < 0:                                        # make the fast upstroke point up
        x, xf = -x, -xf
    f_hr, hr_snr = spectral_peak_fs(xf, fs, (0.7, 3.5), nperseg=2048)
    dist = int(fs * 0.5 / f_hr) if np.isfinite(f_hr) else 30
    peaks, _ = find_peaks(xf, distance=max(dist, 25), prominence=0.3 * np.std(xf))
    if len(peaks) < 20:
        raise ValueError("too few PPG beats")
    feats = {"ppg_hr_spectral": 60 * f_hr, "ppg_hr_snr": hr_snr, "ppg_clipped_frac": float(clipped.mean())}
    feet = np.array([a + np.argmin(xf[a:b]) for a, b in zip(peaks[:-1], peaks[1:])])
    ibi = np.diff(peaks) / fs
    ok = clean_ibi(ibi)
    tb, hr_inst = peaks[1:] / fs, 60 / ibi
    feats.update({"ppg_hr_mean": float(60 / np.mean(ibi[ok])), "ppg_hr_std": float(np.std(hr_inst[ok])),
                  "ppg_beats_ok_frac": float(ok.mean())})
    feats.update({f"ppg_{k}": v for k, v in hrv(ibi[ok]).items()})
    feats["ppg_hr_slope_bpm_per_min"] = float(np.polyfit(tb[ok] / 60, hr_inst[ok], 1)[0]) if ok.sum() > 10 else np.nan
    amp = xf[peaks[1:]] - xf[feet]
    raw_amp = np.array([raw[a:b].max() - raw[a:b].min() for a, b in zip(feet[:-1], feet[1:])])
    feats.update({"ppg_ac_raw": float(np.median(raw_amp)), "ppg_amp_cv": float(np.std(amp) / np.mean(amp))})
    base = bandpass(x[::10], 0.1, 0.8, fs / 10, order=2)
    riiv = (*spectral_peak_fs(base, fs / 10, SENSOR_RESP, nperseg=640), band_centroid(base, fs / 10, nperseg=640))
    riav = resp_from_series(tb, np.where(ok, amp, np.nan))
    rifv = resp_from_series(tb, np.where(ok, ibi, np.nan))
    for name, (fr, frac, cen) in {"riiv": riiv, "riav": riav, "rifv": rifv}.items():
        feats[f"ppg_resp_{name}_bpm"], feats[f"ppg_resp_{name}_power"] = 60 * fr, frac
        feats[f"ppg_resp_{name}_centroid_bpm"] = 60 * cen
    beats, durs = [], []                                             # average beat shape (clipped beats skipped)
    for a, b in zip(feet[:-1], feet[1:]):
        if 0.33 * fs <= b - a <= 2.0 * fs and not clipped[a:b].any():
            seg = xf[a:b + 1]
            seg = (seg - seg.min()) / (np.ptp(seg) + 1e-9)
            beats.append(np.interp(np.linspace(0, len(seg) - 1, 100), np.arange(len(seg)), seg))
            durs.append((b - a) / fs)
    beats = np.array(beats)
    if len(beats) < 10:
        raise ValueError("too few clean PPG beats")
    tmpl = np.median(beats, 0)
    T = float(np.median(durs))
    ip = int(np.argmax(tmpl))
    feats.update({"ppg_beat_duration_s": T, "ppg_rise_time_s": ip / 100 * T,
                  "ppg_template_corr": float(np.mean([np.corrcoef(bt, tmpl)[0, 1] for bt in beats]))})
    for lvl in (0.25, 0.5, 0.75):
        feats[f"ppg_width{int(lvl * 100)}_frac"] = float(np.mean(tmpl > lvl))
    d1 = np.gradient(tmpl)
    d2 = np.gradient(d1)
    mins, _ = find_peaks(-tmpl[ip + 5:90])
    if len(mins):
        n = ip + 5 + mins[0]
        dia = n + int(np.argmax(tmpl[n:90]))
        feats.update({"ppg_notch_found": 1.0, "ppg_notch_height": float(tmpl[n]),
                      "ppg_reflection_index": float(tmpl[dia]), "ppg_delta_t_s": (dia - ip) / 100 * T})
    else:
        feats.update({"ppg_notch_found": 0.0, "ppg_notch_height": np.nan, "ppg_reflection_index": np.nan,
                      "ppg_delta_t_s": np.nan})
    a_i = int(np.argmax(d2[:max(ip, 3)]))
    b_i = a_i + int(np.argmin(d2[a_i:ip + 10])) if ip + 10 > a_i else a_i
    if d2[a_i] > 0:
        feats["ppg_apg_b_a"] = d2[b_i] / d2[a_i]                      # 2nd-derivative wave ratio (linked to vessel ageing)
    for name, v in (("tmpl", tmpl), ("d1", d1 / (np.abs(d1).max() + 1e-9)), ("d2", d2 / (np.abs(d2).max() + 1e-9))):
        for j, val in enumerate(v[::4]):
            feats[f"ppg_{name}_{j:02d}"] = float(val)
    return feats


ECG_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1"]


def ecg_features(path):
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    fs = float(d.get("frequency", 500))
    leads = {L["title"]: np.asarray(L["values"], float) for L in d["data"]}
    n = min(len(v) for k, v in leads.items() if k in ECG_LEADS)
    main = "II" if np.ptp(leads.get("II", [0])) > 0 else max(ECG_LEADS, key=lambda k: np.ptp(leads.get(k, [0])))
    sig = {k: bandpass(leads[k][:n], 0.5, 40.0, fs) for k in ECG_LEADS if k in leads}
    x = sig[main]
    if np.median(x[find_peaks(np.abs(x), distance=int(0.3 * fs))[0]]) < 0:
        x = -x
    energy = np.gradient(bandpass(x, 5.0, 20.0, fs)) ** 2
    pk, _ = find_peaks(energy, distance=int(0.3 * fs), height=0.25 * np.percentile(energy, 99.5))
    w = int(0.05 * fs)
    r = np.unique([max(p - w, 0) + int(np.argmax(x[max(p - w, 0):p + w])) for p in pk])
    if len(r) < 10:
        raise ValueError("too few ECG beats")
    ibi = np.diff(r) / fs
    ok = clean_ibi(ibi)
    feats = {"ecg_hr_mean": float(60 / np.mean(ibi[ok])), "ecg_beats_ok_frac": float(ok.mean())}
    feats.update({f"ecg_{k}": v for k, v in hrv(ibi[ok]).items()})
    pre = int(0.1 * fs)
    for k in ECG_LEADS:
        s = sig.get(k)
        feats[f"ecg_ramp_{k}"] = (float(np.median([s[i] - np.median(s[max(i - pre, 0):max(i - pre // 2, 1)]) for i in r]))
                                  if s is not None and np.ptp(s) > 0 else np.nan)
    lo, hi = int(0.25 * fs), int(0.45 * fs)
    beats = np.array([x[i - lo:i + hi] for i in r if i - lo >= 0 and i + hi <= len(x)])
    tmpl = np.median(beats, 0)
    tmpl = tmpl - np.median(tmpl[:int(0.05 * fs)])
    R = tmpl[lo]
    above = np.where(tmpl > 0.5 * R)[0]
    feats.update({"ecg_r_halfwidth_ms": float((above.max() - above.min()) / fs * 1000) if len(above) else np.nan,
                  "ecg_t_amp_rel": float(tmpl[lo + int(0.15 * fs):lo + int(0.4 * fs)].max() / R),
                  "ecg_st_level_rel": float(tmpl[lo + int(0.08 * fs)] / R),
                  "ecg_template_corr": float(np.mean([np.corrcoef(b, np.median(beats, 0))[0, 1] for b in beats]))})
    for j, val in enumerate((tmpl / (abs(R) + 1e-9))[::10]):
        feats[f"ecg_tmpl_{j:02d}"] = float(val)
    fr, frac, cen = resp_from_series(r[1:] / fs, x[r[1:]])          # breathing from R-wave height changes
    feats.update({"ecg_resp_edr_bpm": 60 * fr, "ecg_resp_edr_power": frac, "ecg_resp_edr_centroid_bpm": 60 * cen})
    return feats


def safe(fn, path):
    try:
        return fn(path)
    except Exception as exc:                                         # a failed file becomes a row of NaN + a flag
        return {"__error__": f"{type(exc).__name__}: {exc}"}

In [ ]:
# ▶ Step 5.3 — Graph: one finger-PPG recording and one ECG, with the detected beats
p_path = PHYSIO_DIR / "ppg" / f"{SESS[0]}.PW"
raw = pd.read_csv(p_path, sep=r"\s+", header=None, usecols=[0]).iloc[:, 0].to_numpy(float)[200:1400]
xf = bandpass(-raw, 0.5, 8.0, 100.0)
pk, _ = find_peaks(xf, distance=40, prominence=0.3 * np.std(xf))
e = json.loads((PHYSIO_DIR / "ecg" / f"{SESS[0]}.json").read_text(encoding="utf-8"))
lead = bandpass(np.asarray([L["values"] for L in e["data"] if L["title"] == "II"][0], float), 0.5, 40.0, 500.0)[:2500]
rp, _ = find_peaks(lead, distance=150, height=0.5 * lead.max())
fig, axes = plt.subplots(2, 1, figsize=(10, 5.2))
axes[0].plot(np.arange(len(xf)) / 100, xf, color=COLORS["cnn"], lw=1.4)
axes[0].scatter(pk / 100, xf[pk], s=24, color=INK, zorder=3)
axes[0].set_title(f"Finger PPG, 12 s ({SESS[0]}): each dot is one heartbeat"); axes[0].set_xlabel("seconds")
axes[1].plot(np.arange(len(lead)) / 500, lead, color=COLORS["multimodal_specialists"], lw=1.2)
axes[1].scatter(rp / 500, lead[rp], s=24, color=INK, zorder=3)
axes[1].set_title("ECG lead II, 5 s: each dot is an R wave"); axes[1].set_xlabel("seconds")
fig.tight_layout(); savefig(fig, "06_sensor_examples"); plt.show()

In [ ]:
# ▶ Step 5.4 — PPG and ECG features for all 1,200 sessions (computed once, then reused)
def build_sensor(kind, fn, ext):
    path = FEATURES / f"{kind}.npz"
    if RECOMPUTE_PHYSIO or not path.exists():
        t0 = time.time()
        rows = Parallel(n_jobs=4)(delayed(safe)(fn, PHYSIO_DIR / kind / f"{s}.{ext}") for s in SESS)
        names = sorted({k for r in rows for k in r if k != "__error__"})
        X = np.array([[r.get(k, np.nan) for k in names] for r in rows], dtype=float)
        avail = np.array([0.0 if "__error__" in r else 1.0 for r in rows])
        X[avail == 0] = np.nan
        save_features(path, SESS, np.column_stack([X, avail]), names + [f"{kind}_available"])
        print(f"{kind}: computed in {time.time() - t0:.0f}s, failed files: {int((avail == 0).sum())}")
    with np.load(path) as z:
        names = list(z["names"])
    return load_features(path, SESS), names


X_ppg, ppg_names = build_sensor("ppg", ppg_features, "PW")
X_ecg, ecg_names = build_sensor("ecg", ecg_features, "json")
print("PPG features:", X_ppg.shape, "| ECG features:", X_ecg.shape)
BLOCKS = {"dino": X_dino, "clip": X_clip, "temporal": X_rppg, "ppg": X_ppg, "ecg": X_ecg}   # every feature block

In [ ]:
# ▶ Step 5.5 — Graph: the camera missed the exercise
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
hr_ppg = X_ppg[:, ppg_names.index("ppg_hr_mean")]
timing = {"Cuff pulse, right after squats": (df.pulse[after == 0].mean(), df.pulse[after == 1].mean()),
          "Heart rate during the video (finger PPG)": (np.nanmean(hr_ppg[after == 0]), np.nanmean(hr_ppg[after == 1]))}


def state_auc(block_names):
    """How well can these inputs tell 'after exercise' from 'before'? (ROC AUC; 0.5 = coin flip)"""
    from sklearn.linear_model import LogisticRegression
    X = np.column_stack([BLOCKS[b] for b in block_names])
    X = np.where(np.isfinite(X), X, np.nanmedian(X, 0))
    fold, p = session_folds(0), np.zeros(len(df))
    for k in range(N_FOLDS):                                     # people-split folds, as everywhere else
        tr, te = fold != k, fold == k
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(C=0.01, max_iter=3000).fit(sc.transform(X[tr]), after[tr])
        p[te] = clf.predict_proba(sc.transform(X[te]))[:, 1]
    return roc_auc_score(after, p)


auc = {"Face video (all features)": state_auc(["dino", "clip", "temporal"]), "rPPG colour signals": state_auc(["temporal"]),
       "Finger PPG": state_auc(["ppg"]), "ECG": state_auc(["ecg"]), "PPG + ECG": state_auc(["ppg", "ecg"])}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.1, 1]})
for i, (lab, (b, a)) in enumerate(timing.items()):
    a1.plot([b, a], [i, i], color=GRID, lw=3, zorder=1)
    a1.scatter([b], [i], s=70, color=COLORS["constant"], zorder=2)
    a1.scatter([a], [i], s=80, color="#c23a4e", zorder=3)
    a1.text(a + 0.6, i, f"{signed(a - b, 1)} bpm", va="center", fontsize=10, color=INK)
a1.set_yticks(range(len(timing)), list(timing)); a1.set_ylim(-0.6, 1.6); a1.invert_yaxis()
a1.set_xlabel("beats per minute (grey = before, red = after exercise)"); a1.set_title("Heart rate: labels vs video")
a2.barh(list(auc), [v - 0.5 for v in auc.values()], left=0.5, height=0.55,
        color=[COLORS["physio_only"] if ("Finger" in k or "ECG" in k) else COLORS["video_specialists"] for k in auc])
for i, v in enumerate(auc.values()):
    a2.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=9, color=INK)
a2.set_xlim(0.5, 1.0); a2.invert_yaxis(); a2.set_xlabel("ROC AUC (0.5 = coin flip, 1 = perfect)")
a2.set_title("Can a model tell before from after exercise?")
fig.tight_layout(); savefig(fig, "07_timing_mismatch"); plt.show()

**What it means.** Right after the squats, the cuff pulse is about 20 bpm higher. During the three-minute video it is
only about 4 bpm higher: the heart had mostly recovered before filming. So the labels that change with exercise
(breathing rate, pulse, stiffness) describe a moment the camera did not see. From the video, a model can barely tell
the two sessions apart. This caps what any video model can learn for those targets.

## Step 6 · Baseline 2: your CNN

**What.** Your CNN from `Face_AI/baseline_test`, with the same layers and settings, trained once inside each of the 5
folds of shuffle 0. A shared encoder reads each of the 8 face crops, the 8 results are averaged, and two heads predict
the 9 numbers and the sex. **Why.** It is the "before" model. Retraining it in the same folds makes the comparison
fair. The full run takes about 2.5 hours on a CPU, so by default the saved run is loaded.

In [ ]:
# ▶ Step 6.1 — The CNN: 4 convolution blocks -> average over 8 frames -> 9 numbers + 1 sex score (533,075 weights)
import torch.nn as nn
import torch.nn.functional as F
CNN_EPOCHS, CNN_BATCH, CNN_LR, CNN_WD = 30, 16, 1e-3, 1e-3


def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.MaxPool2d(2))


class FaceCNN(nn.Module):
    def __init__(self, n_reg=len(REG_TARGETS)):
        super().__init__()
        self.encoder = nn.Sequential(conv_block(3, 24), conv_block(24, 48), conv_block(48, 96), conv_block(96, 160),
                                     nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.dropout = nn.Dropout(0.3)
        self.reg_head, self.sex_head = nn.Linear(160, n_reg), nn.Linear(160, 1)

    def forward(self, x):                                            # x: (batch, frames, 3, H, W) in [0, 1]
        b, t = x.shape[:2]
        feats = self.dropout(self.encoder(x.flatten(0, 1)).view(b, t, -1).mean(1))   # average the 8 frames
        return self.reg_head(feats), self.sex_head(feats).squeeze(1)


def augment(x):
    """Flips and small shifts only. No colour changes: skin colour carries the pulse signal."""
    flip = torch.rand(x.shape[0], device=x.device) < 0.5
    x[flip] = x[flip].flip(-1)
    dx, dy = np.random.randint(-6, 7, size=2)
    return torch.roll(x, shifts=(int(dy), int(dx)), dims=(-2, -1))


def masked_loss(r, s, yr, ys):
    m, ms = ~torch.isnan(yr), ~torch.isnan(ys)
    reg = F.mse_loss(r[m], yr[m]) if m.any() else r.sum() * 0
    sex = F.binary_cross_entropy_with_logits(s[ms], ys[ms]) if ms.any() else s.sum() * 0
    n = int(m.any(0).sum())
    return (reg * n + sex) / (n + 1)


@torch.no_grad()
def cnn_predict(model, x, batch=64):
    model.eval()
    regs, sexes = [], []
    for i in range(0, len(x), batch):
        r, s = model(x[i:i + batch].to(DEVICE).float().div_(255.0))
        regs.append(r.cpu()); sexes.append(torch.sigmoid(s).cpu())
    return torch.cat(regs).numpy(), torch.cat(sexes).numpy()


def train_cnn_fold(k, x_all, epochs=CNN_EPOCHS, people_limit=None, history=None):
    """Train on folds != k (15% of those people held back to pick the best epoch); predict fold k."""
    fold = session_folds(0)
    tr_all, te = np.where(fold != k)[0], np.where(fold == k)[0]
    pp = df.iloc[tr_all].drop_duplicates("patient_id")
    _, va_p = next(StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED + 100 + k).split(pp, pp["sex"], groups=pp["patient_id"]))
    is_va = df["patient_id"].iloc[tr_all].isin(set(pp["patient_id"].iloc[va_p])).to_numpy()
    tr, va = tr_all[~is_va], tr_all[is_va]
    if people_limit:                                                  # quick check: a small subset only
        keep = set(df["patient_id"].iloc[tr].unique()[:people_limit])
        tr = tr[df["patient_id"].iloc[tr].isin(keep).to_numpy()]
    Y = df[REG_TARGETS].to_numpy(float)
    mu, sd = np.nanmean(Y[tr], 0), np.nanstd(Y[tr], 0)               # standardise with training people only
    yr = torch.tensor((Y - mu) / sd, dtype=torch.float32)
    ys = torch.tensor(df["sex_label"].to_numpy(float), dtype=torch.float32)
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    model = FaceCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CNN_LR, weight_decay=CNN_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, best_state = np.inf, None
    for epoch in range(1, epochs + 1):
        model.train(); t0, losses = time.time(), []
        for idx in torch.from_numpy(tr)[torch.randperm(len(tr))].split(CNN_BATCH):
            r, s = model(augment(x_all[idx].to(DEVICE).float().div_(255.0)))
            loss = masked_loss(r, s, yr[idx].to(DEVICE), ys[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        sched.step()
        rv, sv = cnn_predict(model, x_all[va])
        val = masked_loss(torch.from_numpy(rv), torch.logit(torch.from_numpy(sv).clamp(1e-6, 1 - 1e-6)), yr[va], ys[va]).item()
        print(f"fold {k} epoch {epoch:2d}  train {np.mean(losses):.4f}  validation {val:.4f}  ({time.time() - t0:.0f}s)")
        if history is not None:
            history.append({"fold": k, "epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val})
        if val < best:
            best, best_state = val, {n: v.detach().clone() for n, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    r, s = cnn_predict(model, x_all[te])
    return te, r * sd + mu, s                                        # back to real units


print("weights:", sum(p.numel() for p in FaceCNN().parameters()))

In [ ]:
# ▶ Step 6.2 — Train the CNN in the 5 folds (or load the saved run)
if RECOMPUTE_CNN or CNN_QUICK_CHECK:
    with np.load(FACES_NPZ) as z:
        x_all = torch.from_numpy(z["frames"]).permute(0, 1, 4, 2, 3).contiguous()   # (1200, 8, 3, 112, 112) uint8
if CNN_QUICK_CHECK:
    print("Quick check: 1 epoch, 60 training people (nothing is saved)")
    train_cnn_fold(0, x_all, epochs=1, people_limit=60)
if RECOMPUTE_CNN:
    P, history = np.full((len(df), len(ALL_TARGETS)), np.nan), []
    for k in range(N_FOLDS):
        te, r, s = train_cnn_fold(k, x_all, history=history)
        P[te, :len(REG_TARGETS)], P[te, -1] = r, s
    save_oof("cnn", P, 0)
    pd.DataFrame(history).to_csv(LOGS / "cnn_history.csv", index=False)
cnn_oof = load_oof("cnn")
print("CNN predictions available for", 0 if cnn_oof is None else len(cnn_oof), "sessions (shuffle 0)")

In [ ]:
# ▶ Step 6.3 — Graph: the CNN's training and validation loss, one line per fold
hist = pd.read_csv(LOGS / "cnn_history.csv")
fig, ax = plt.subplots(figsize=(8, 3.6))
for k, g in hist.groupby("fold"):
    ax.plot(g.epoch, g.train_loss, color=COLORS["mtl_video_all"], lw=1.2, alpha=0.8, label="training loss" if k == 0 else None)
    ax.plot(g.epoch, g.val_loss, color=COLORS["cnn"], lw=1.2, alpha=0.8, label="validation loss" if k == 0 else None)
ax.set_xlabel("epoch"); ax.set_ylabel("loss (standardised targets)"); ax.legend()
ax.set_title("Your CNN: training loss keeps falling, validation loss is noisy and flattens")
fig.tight_layout(); savefig(fig, "08_cnn_training_curves"); plt.show()

## Step 7 · The improved model: a specialist for every target

**What.** For each target separately, the **inner loop** tries:

* **feature sets ("routes"):** DINOv2, CLIP, both, the pulse signals, all video features, video + predicted
  age/sex/BMI, and with contact sensors: PPG, ECG, both, and video + sensors;
* **model families:** ridge regression, partial least squares (PLS), kernel ridge (non-linear), shallow boosted trees,
  and **the median guess itself**;
* **penalties** for each family.

It keeps whatever scores best on the inner check and refits it on all training people.

**Why.**

* Different targets need different tools.
* Strong penalties stop the model from memorising 480 people (Step 11 shows this).
* The median guess as a candidate lets the model **abstain** when the video has nothing to offer.

**Two details that matter.**

* **Aim at the median.** Squared-error models predict an average, but our score rewards the median. So each model's
  output is shifted by the median of its training errors.
* **Equal weight per feature set.** Each set of features is scaled to the same total weight, so 768 CLIP numbers
  cannot drown 58 ECG numbers.

In [ ]:
# ▶ Step 7.1 — The candidate models (fast closed-form solutions, so the whole nested search runs on a CPU)
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
RIDGE_ALPHAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]     # x number of training sessions
PLS_KS = [1, 2, 4, 8, 16]
KRR_LAMBDAS = [0.1, 1.0, 10.0, 100.0]
INNER_FOLDS, CLIP_Z = 4, 6.0
HGB_ROUTES = {"temporal", "physio"}                    # trees only on the smaller signal feature sets


class BlockScaler:
    """Fill gaps with the training median, standardise, clip at ±6 SD, and give every feature block equal weight."""
    def __init__(self, sizes):
        self.sizes = sizes

    def fit(self, X):
        med = np.nanmedian(X, 0)
        self.med = np.where(np.isfinite(med), med, 0.0)
        Xi = np.where(np.isnan(X), self.med, X)
        self.mu, sd = Xi.mean(0), Xi.std(0)
        self.keep = sd > 1e-9
        self.sd = np.where(self.keep, sd, 1.0)
        self.w, start = np.ones(X.shape[1]), 0
        for size in self.sizes:
            sl = slice(start, start + size)
            self.w[sl] = 1.0 / np.sqrt(max(int(self.keep[sl].sum()), 1))
            start += size
        return self

    def transform(self, X):
        Xi = np.where(np.isnan(X), self.med, X)
        return (np.clip((Xi - self.mu) / self.sd, -CLIP_Z, CLIP_Z) * self.w)[:, self.keep]


def ridge_all(Ztr, Ytr, Zte):
    """Ridge regression for every penalty at once, from one SVD of the training features."""
    zm, ym = Ztr.mean(0), Ytr.mean(0)
    U, s, Vt = np.linalg.svd(Ztr - zm, full_matrices=False)
    UtY, Zc = U.T @ (Ytr - ym), Zte - zm
    return {f"ridge_a{a:g}": Zc @ (Vt.T @ ((s / (s ** 2 + a * len(Ztr)))[:, None] * UtY)) + ym for a in RIDGE_ALPHAS}


def pls_all(Ztr, Ytr, Zte):
    """Partial least squares (one target at a time): a few directions that best predict y."""
    zm = Ztr.mean(0)
    X0, Xt = Ztr - zm, Zte - zm
    kmax = min(max(PLS_KS), X0.shape[1])
    out = {f"pls_k{k}": np.zeros((len(Zte), Ytr.shape[1])) for k in PLS_KS}
    for j in range(Ytr.shape[1]):
        ym = Ytr[:, j].mean()
        yc, X, W, P_, Q = Ytr[:, j] - ym, X0.copy(), [], [], []
        for _ in range(kmax):
            w = X.T @ yc
            nw = np.linalg.norm(w)
            if nw < 1e-12:
                break
            w /= nw
            t = X @ w
            tt = t @ t
            p, q = X.T @ t / tt, (yc @ t) / tt
            X -= np.outer(t, p)
            yc = yc - q * t
            W.append(w); P_.append(p); Q.append(q)
        W, P_, Q = np.array(W).T, np.array(P_).T, np.array(Q)
        for k in PLS_KS:
            kk = min(k, W.shape[1])
            out[f"pls_k{k}"][:, j] = Xt @ (W[:, :kk] @ np.linalg.solve(P_[:, :kk].T @ W[:, :kk], Q[:kk])) + ym
    return out


def krr_all(Ztr, Ytr, Zte):
    """Kernel ridge with an RBF kernel (a smooth non-linear model), every penalty from one eigendecomposition."""
    gamma = 1.0 / (2.0 * Ztr.var(0).sum() + 1e-12)
    sq = lambda A, B: np.maximum((A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T, 0)
    K, Kt, ym = np.exp(-gamma * sq(Ztr, Ztr)), np.exp(-gamma * sq(Zte, Ztr)), Ytr.mean(0)
    lam, Qm = np.linalg.eigh(K)
    QtY = Qm.T @ (Ytr - ym)
    return {f"krr_l{l:g}": Kt @ (Qm @ (QtY / (lam + l)[:, None])) + ym for l in KRR_LAMBDAS}


def hgb_all(Xtr, Ytr, Xte, sex_mask_tr):
    """Shallow boosted trees, absolute-error loss (leaves hold medians, matching the MAE score)."""
    pred = np.zeros((len(Xte), Ytr.shape[1]))
    kw = dict(max_iter=150, learning_rate=0.05, max_depth=3, min_samples_leaf=20, l2_regularization=1.0,
              early_stopping=False, random_state=0)
    for j, t in enumerate(ALL_TARGETS):
        if t == "sex":
            m = sex_mask_tr
            pred[:, j] = HistGradientBoostingClassifier(**kw).fit(Xtr[m], Ytr[m, j].round().astype(int)).predict_proba(Xte)[:, 1]
        else:
            pred[:, j] = HistGradientBoostingRegressor(loss="absolute_error", **kw).fit(Xtr, Ytr[:, j]).predict(Xte)
    return {"hgb": pred}


def fit_predict_all(Xtr, Ytr, Xte, sizes, use_hgb, sex_mask_tr):
    """Predictions of every candidate for every target. 'const' (the median guess) is a candidate too."""
    sc = BlockScaler(sizes).fit(Xtr)
    Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    both, out = np.vstack([Zte, Ztr]), {}
    for fn in (ridge_all, pls_all, krr_all):
        for cfg, P in fn(Ztr, Ytr, both).items():
            Pte, Ptr = P[:len(Zte)], P[len(Zte):]
            shift = np.median(Ytr - Ptr, axis=0)                     # aim at the median of the training errors
            shift[-1] = 0.0                                          # (not for the sex probability)
            out[cfg] = Pte + shift
    if use_hgb:
        out.update(hgb_all(Xtr, Ytr, Xte, sex_mask_tr))
    const = np.median(Ytr, axis=0)
    const[-1] = Ytr[sex_mask_tr, -1].mean() if sex_mask_tr.any() else 0.5
    out["const"] = np.tile(const, (len(Xte), 1))
    return out


def target_matrix():
    """Targets in the space the models are fitted in: log for age and BMI, sex as 0/1."""
    Y = np.column_stack([df[REG_TARGETS].to_numpy(float), df["sex_label"].to_numpy(float)])
    for j, t in enumerate(ALL_TARGETS):
        if t in LOG_TARGETS:
            Y[:, j] = np.log(Y[:, j])
    return Y


def back_transform(P):
    P = P.copy()
    for j, t in enumerate(ALL_TARGETS):
        if t in LOG_TARGETS:
            P[:, j] = np.exp(P[:, j])
        if t == "sex":
            P[:, j] = np.clip(P[:, j], 0.01, 0.99)
    return P


Y_fit = target_matrix()
print("candidates per feature set:", len(RIDGE_ALPHAS) + len(PLS_KS) + len(KRR_LAMBDAS) + 1, "(+1 tree model on signal sets)")

In [ ]:
# ▶ Step 7.2 — Check our maths: the fast formulas must equal scikit-learn's models
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
rng = np.random.default_rng(0)
Xa, Ya, Xb = rng.normal(size=(200, 50)), rng.normal(size=(200, 3)), rng.normal(size=(40, 50))
Ya = Xa[:, :5] @ rng.normal(size=(5, 3)) + Ya
ours = ridge_all(Xa, Ya, Xb)
assert all(np.allclose(ours[f"ridge_a{a:g}"], Ridge(alpha=a * 200).fit(Xa, Ya).predict(Xb), atol=1e-8) for a in RIDGE_ALPHAS)
ours = pls_all(Xa, Ya, Xb)
assert all(np.allclose(ours[f"pls_k{k}"][:, j], PLSRegression(n_components=k, scale=False).fit(Xa, Ya[:, j]).predict(Xb).ravel(), atol=1e-6)
           for k in PLS_KS for j in range(3))
ours, g = krr_all(Xa, Ya, Xb), 1.0 / (2.0 * Xa.var(0).sum())
assert all(np.allclose(ours[f"krr_l{l:g}"], KernelRidge(alpha=l, kernel="rbf", gamma=g).fit(Xa, Ya - Ya.mean(0)).predict(Xb) + Ya.mean(0), atol=1e-6)
           for l in KRR_LAMBDAS)
print("All three fast formulas match scikit-learn.")

In [ ]:
# ▶ Step 7.3 — Watch the inner loop choose: one outer fold, the face features (DINOv2 + CLIP), two targets
def inner_splits(pids_tr, seed):
    groups = np.array_split(np.random.default_rng(seed).permutation(np.unique(pids_tr)), INNER_FOLDS)
    return [np.isin(pids_tr, g) for g in groups]


def score(pred, y, p, target):
    """Selection score (lower = better): MAE averaged per person; Brier score for sex."""
    ok = np.isfinite(y)
    err = (pred[ok] - y[ok]) ** 2 if target == "sex" else np.abs(pred[ok] - y[ok])
    return float(pd.Series(err).groupby(p[ok]).mean().mean())


fold = session_folds(0)
tr = np.where(fold != 0)[0]
X = np.column_stack([X_dino, X_clip])
inner = {}
for va_mask in inner_splits(pids[tr], SEED):
    itr, iva = tr[~va_mask], tr[va_mask]
    Yi = Y_fit[itr].copy()
    Yi[~np.isfinite(Yi[:, -1]), -1] = np.nanmean(Yi[:, -1])
    for cfg, P in fit_predict_all(X[itr], Yi, X[iva], [384, 768], False, np.isfinite(Y_fit[itr, -1])).items():
        inner.setdefault(cfg, np.zeros((len(tr), len(ALL_TARGETS))))[va_mask] = back_transform(P)
for t in ["age", "temperature"]:
    j = ALL_TARGETS.index(t)
    sc = pd.Series({cfg: score(P[:, j], Y_nat[tr, j], pids[tr], t) for cfg, P in inner.items()}).sort_values()
    display(Markdown(f"**{NAME[t]}**: best = `{sc.index[0]}`; the median guess (`const`) scores {sc['const']:.3f}"))
    display(sc.head(6).rename("inner MAE (lower is better)").to_frame().T.round(3))

**What you should see.** For **age**, a regularised model on the face features clearly beats the median guess. For
**temperature**, nothing beats `const`, so the specialist abstains and predicts the median. That is the behaviour we
want: no fake skill.

In [ ]:
# ▶ Step 7.4 — The full nested cross-validation (outer folds score, inner folds choose)
VIDEO_ROUTES = {"dino": ["dino"], "clip": ["clip"], "appearance": ["dino", "clip"], "temporal": ["temporal"],
                "video_all": ["dino", "clip", "temporal"]}
PHYSIO_ROUTES = {"ppg": ["ppg"], "ecg": ["ecg"], "physio": ["ppg", "ecg"], "video+physio": ["dino", "clip", "temporal", "ppg", "ecg"]}
CHAIN_VIDEO = {"video_all+demo": ["dino", "clip", "temporal", "demo"], "demo": ["demo"]}   # + predicted age/BMI/sex
CHAIN_MM = {"video+physio+demo": ["dino", "clip", "temporal", "ppg", "ecg", "demo"]}
TIERS = {"video_specialists": list(VIDEO_ROUTES) + list(CHAIN_VIDEO),
         "physio_only": ["ppg", "ecg", "physio"],
         "multimodal_specialists": list(VIDEO_ROUTES) + list(CHAIN_VIDEO) + list(PHYSIO_ROUTES) + list(CHAIN_MM)}


def run_route(route, block_names, fold, repeat, extra=None, outer=range(N_FOLDS)):
    """One feature route: choose a candidate per (outer fold, target) on the inner folds, then predict the outer fold."""
    oof = np.full((len(df), len(ALL_TARGETS)), np.nan)
    inner_best, log = {}, []
    for k in outer:
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
        mats = [extra[k] if b == "demo" else BLOCKS[b] for b in block_names]
        X, sizes = np.column_stack(mats), [m.shape[1] for m in mats]
        Ytr = Y_fit[tr].copy()
        sex_ok = np.isfinite(Ytr[:, -1])
        Ytr[~sex_ok, -1] = np.nanmean(Ytr[:, -1])
        inner = {}
        for va_mask in inner_splits(pids[tr], SEED + 1000 * repeat + 10 * k):
            itr, iva = tr[~va_mask], tr[va_mask]
            preds = fit_predict_all(X[itr], Ytr[~va_mask], X[iva], sizes, route in HGB_ROUTES, np.isfinite(Y_fit[itr, -1]))
            for cfg, P in preds.items():
                inner.setdefault(cfg, np.zeros((len(tr), len(ALL_TARGETS))))[va_mask] = back_transform(P)
        final = fit_predict_all(X[tr], Ytr, X[te], sizes, route in HGB_ROUTES, sex_ok)
        for j, t in enumerate(ALL_TARGETS):
            scores = {cfg: score(P[:, j], Y_nat[tr, j], pids[tr], t) for cfg, P in inner.items()}
            best = min(scores, key=scores.get)
            oof[te, j] = back_transform(final[best])[:, j]
            inner_best[(k, t)] = inner[best][:, j]
            log.append({"repeat": repeat, "fold": k, "route": route, "target": t, "config": best, "inner_score": scores[best]})
    return oof, pd.DataFrame(log), inner_best


def select_tier(logs, routes, repeat):
    lg = logs[(logs["repeat"] == repeat) & logs["route"].isin(routes)]
    return lg.loc[lg.groupby(["fold", "target"])["inner_score"].idxmin()]   # best inner score per (fold, target)


def assemble(best, route_oof, fold):
    out = np.full((len(df), len(ALL_TARGETS)), np.nan)
    for row in best.itertuples():
        out[fold == row.fold, ALL_TARGETS.index(row.target)] = route_oof[row.route][fold == row.fold, ALL_TARGETS.index(row.target)]
    return out


def run_nested_cv(repeat):
    fold, route_oof, inner_store, logs = session_folds(repeat), {}, {}, []
    for route, blist in {**VIDEO_ROUTES, **PHYSIO_ROUTES}.items():
        t0 = time.time()
        oof, log, ib = run_route(route, blist, fold, repeat)
        route_oof[route], inner_store[route] = oof, ib
        logs.append(log); save_oof(f"route_{route}", oof, repeat)
        print(f"  shuffle {repeat}  route {route:13s} {time.time() - t0:5.0f}s")
    L = pd.concat(logs, ignore_index=True)
    stage1 = select_tier(L, list(VIDEO_ROUTES), repeat)                # stacking: predicted age, BMI, sex as new inputs
    demo_blocks = {}
    for k in range(N_FOLDS):
        D = np.zeros((len(df), 3))
        for c, t in enumerate(["age", "bmi", "sex"]):
            rt = stage1[(stage1["fold"] == k) & (stage1["target"] == t)]["route"].iloc[0]
            D[fold != k, c] = inner_store[rt][(k, t)]                  # inner (honest) predictions for training people
            D[fold == k, c] = route_oof[rt][fold == k, ALL_TARGETS.index(t)]
        demo_blocks[k] = D
    for route, blist in {**CHAIN_VIDEO, **CHAIN_MM}.items():
        t0 = time.time()
        oof, log, _ = run_route(route, blist, fold, repeat, extra=demo_blocks)
        route_oof[route] = oof
        logs.append(log); save_oof(f"route_{route}", oof, repeat)
        print(f"  shuffle {repeat}  route {route:13s} {time.time() - t0:5.0f}s")
    L = pd.concat(logs, ignore_index=True)
    L.to_csv(RESULTS / f"selection_log_r{repeat}.csv", index=False)
    for tier, routes in TIERS.items():
        best = select_tier(L, routes, repeat)
        best.assign(tier=tier).to_csv(OOF / f"selection_{tier}_r{repeat}.csv", index=False)
        save_oof(tier, assemble(best, route_oof, fold), repeat)

In [ ]:
# ▶ Step 7.5 — Run the nested cross-validation for the 3 shuffles (or load the saved results)
if RECOMPUTE_SPECIALISTS:
    for r in range(N_REPEATS):
        t0 = time.time()
        run_nested_cv(r)
        print(f"shuffle {r} done in {(time.time() - t0) / 60:.1f} min")
missing = [n for n in TIERS if load_oof(n) is None]
print("saved specialist results:", "complete" if not missing else f"MISSING {missing}: set RECOMPUTE_SPECIALISTS = True")

## Step 8 · One shared network, or separate models?

**What.** **Multi-task learning** trains *one* network for all 10 targets. Its first layers are shared, and each
target has its own small output. We train it on the same features as the specialists. As a **control**, we also train
the *same* network once per target, so any difference comes from the sharing alone.

**Why.** Sharing can help related targets learn from each other. It can also hurt when weak, noisy targets pull the
shared layers away from the strong ones (**negative transfer**). Only a test can tell which happens here.

In [ ]:
# ▶ Step 8.1 — The network: shared layers -> one output per target, with learned task weights
MLP_SEEDS, MLP_EPOCHS, MLP_PATIENCE, MLP_BATCH = 5, 300, 30, 64


class MTLNet(nn.Module):
    def __init__(self, p, n_reg, with_sex):
        super().__init__()
        self.trunk = nn.Sequential(nn.Dropout(0.2), nn.Linear(p, 256), nn.GELU(), nn.Dropout(0.5),
                                   nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3))      # shared layers
        self.reg = nn.Linear(128, n_reg) if n_reg else None
        self.sex = nn.Linear(128, 1) if with_sex else None
        self.log_var = nn.Parameter(torch.zeros(n_reg + int(with_sex)))   # one learned weight per task (Kendall 2018)

    def forward(self, x):
        h = self.trunk(x)
        r = self.reg(h) if self.reg is not None else x.new_zeros(len(x), 0)
        s = self.sex(h).squeeze(1) if self.sex is not None else x.new_zeros(len(x))
        return r, s


def task_losses(r, s, yr, ys, with_sex):
    L = [F.mse_loss(r[:, j], yr[:, j]) for j in range(r.shape[1])]
    if with_sex:
        m = ~torch.isnan(ys)
        L.append(F.binary_cross_entropy_with_logits(s[m], ys[m]))
    return torch.stack(L)


def train_mlp(Z, Yr, ys, with_sex, seed, curve=None):
    """Train with early stopping: keep the epoch with the lowest validation loss."""
    torch.manual_seed(seed)
    model = MTLNet(Z["tr"].shape[1], Yr["tr"].shape[1], with_sex).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    best, best_state, bad = np.inf, None, 0
    for epoch in range(MLP_EPOCHS):
        model.train()
        tl = []
        for idx in torch.randperm(len(Z["tr"])).split(MLP_BATCH):
            r, s = model(Z["tr"][idx])
            L = task_losses(r, s, Yr["tr"][idx], ys["tr"][idx], with_sex)
            loss = (torch.exp(-model.log_var) * L + model.log_var).sum() / 2
            opt.zero_grad(); loss.backward(); opt.step(); tl.append(L.mean().item())
        model.eval()
        with torch.no_grad():
            val = task_losses(*model(Z["va"]), Yr["va"], ys["va"], with_sex).mean().item()
        if curve is not None:
            curve.append({"epoch": epoch + 1, "train_loss": float(np.mean(tl)), "val_loss": val, "seed": seed})
        if val < best - 1e-4:
            best, best_state, bad = val, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= MLP_PATIENCE:
                break
    model.load_state_dict(best_state)
    return model


def run_mlp(route, single_task):
    names = {**VIDEO_ROUTES, **PHYSIO_ROUTES}[route]
    X, sizes = np.column_stack([BLOCKS[b] for b in names]), [BLOCKS[b].shape[1] for b in names]
    Y = np.array(df[REG_TARGETS], dtype=float)
    for j, t in enumerate(REG_TARGETS):
        if t in LOG_TARGETS:
            Y[:, j] = np.log(Y[:, j])
    ysex = df["sex_label"].to_numpy(float)
    tag = ("stl_" if single_task else "mtl_") + route.replace("+", "_")
    groups = [[j] for j in range(len(ALL_TARGETS))] if single_task else [list(range(len(ALL_TARGETS)))]
    curves = []
    for r in ([0] if single_task else range(N_REPEATS)):
        fold = session_folds(r)
        oof = np.full((len(df), len(ALL_TARGETS)), np.nan)
        for k in range(N_FOLDS):
            t0 = time.time()
            tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
            rng = np.random.default_rng(SEED + 77 * r + k)
            va_people = rng.choice(np.unique(pids[tr]), size=int(0.15 * len(np.unique(pids[tr]))), replace=False)
            is_va = np.isin(pids[tr], va_people)
            itr, iva = tr[~is_va], tr[is_va]
            sc = BlockScaler(sizes).fit(X[itr])
            Z = {n_: torch.tensor(sc.transform(X[i]), dtype=torch.float32, device=DEVICE) for n_, i in (("tr", itr), ("va", iva), ("te", te))}
            mu, sd = Y[itr].mean(0), Y[itr].std(0)
            for group in groups:
                reg_cols = [j for j in group if j < len(REG_TARGETS)]
                with_sex = len(REG_TARGETS) in group
                Yr = {n_: torch.tensor((Y[i][:, reg_cols] - mu[reg_cols]) / sd[reg_cols], dtype=torch.float32, device=DEVICE)
                      for n_, i in (("tr", itr), ("va", iva))}
                ys = {n_: torch.tensor(ysex[i], dtype=torch.float32, device=DEVICE) for n_, i in (("tr", itr), ("va", iva))}
                pr, ps = [], []
                for seed in range(MLP_SEEDS if not single_task else 3):
                    m = train_mlp(Z, Yr, ys, with_sex, SEED + seed, curves if (r == 0 and k == 0 and not single_task) else None)
                    m.eval()
                    with torch.no_grad():
                        a, b = m(Z["te"])
                    pr.append(a.cpu().numpy()); ps.append(torch.sigmoid(b).cpu().numpy())
                pr = np.mean(pr, 0) * sd[reg_cols] + mu[reg_cols]
                for c, j in enumerate(reg_cols):
                    oof[te, j] = np.exp(pr[:, c]) if ALL_TARGETS[j] in LOG_TARGETS else pr[:, c]
                if with_sex:
                    oof[te, -1] = np.clip(np.mean(ps, 0), 0.01, 0.99)
            print(f"  {tag} shuffle {r} fold {k}: {time.time() - t0:.0f}s")
        save_oof(tag, oof, r)
    if curves:
        pd.DataFrame(curves).to_csv(LOGS / f"{tag}_curves.csv", index=False)

In [ ]:
# ▶ Step 8.2 — Train the networks (or load the saved results)
if RECOMPUTE_MLP:
    run_mlp("video_all", single_task=False)       # one shared network, video features
    run_mlp("video+physio", single_task=False)    # one shared network, video + sensors
    run_mlp("video_all", single_task=True)        # control: the same network, one target at a time (shuffle 0)
print("saved network results:", {n: load_oof(n) is not None for n in ["mtl_video_all", "mtl_video_physio", "stl_video_all"]})

In [ ]:
# ▶ Step 8.3 — Graph: early stopping keeps the epoch where validation loss is lowest
mtl_curves = pd.read_csv(LOGS / "mtl_video_all_curves.csv")          # one row per epoch and seed (fold 0)
fig, ax = plt.subplots(figsize=(8, 3.6))
for s, g in mtl_curves.groupby("seed"):
    first = s == mtl_curves.seed.min()
    ax.plot(g.epoch, g.train_loss, color=COLORS["mtl_video_all"], lw=1.2, alpha=0.8, label="training loss" if first else None)
    ax.plot(g.epoch, g.val_loss, color=COLORS["video_specialists"], lw=1.2, alpha=0.8, label="validation loss" if first else None)
    b = g.loc[g.val_loss.idxmin()]
    ax.scatter([b.epoch], [b.val_loss], s=40, color=COLORS["video_specialists"], edgecolor=SURFACE, zorder=3)
ax.set_xlabel("epoch"); ax.set_ylabel("average task loss"); ax.legend()
ax.set_title("Shared network: validation loss bottoms out early, then rises while training loss keeps falling")
fig.tight_layout(); savefig(fig, "09_mtl_early_stopping"); plt.show()

**What it means.** After the dot, the network keeps getting better on its training people (green) and worse on new
people (blue). That is overfitting in action. Early stopping keeps the network from the dot.

## Step 9 · Evaluation

**What.** For every model and target, the error is computed **per person first**, then averaged over people, so
somebody with two sessions does not count twice. The table reports:

* **MAE** in the target's own unit, and **skill** against the median guess;
* **RMSE** and **R²**;
* **correlation r**, and the **calibration slope**: 1 means the predictions are spread correctly, more than 1 means
  they are squeezed towards the middle.

For sex we report accuracy, **balanced accuracy** (the average of the per-sex accuracies) and **ROC AUC**.

**Uncertainty.** We resample whole **people** 2,000 times (bootstrap). The 2.5th–97.5th percentiles give a 95%
confidence interval. Comparisons between two models are **paired**: the same people in each resample.

In [ ]:
# ▶ Step 9.1 — Metric functions
from scipy.stats import pearsonr
from sklearn.metrics import balanced_accuracy_score, roc_curve, auc as auc_area, confusion_matrix
B, RNG = 2000, np.random.default_rng(20261013)
truth = df.set_index("key")[REG_TARGETS + ["sex_label", "step", "patient_id"]]


def person_errors(pred, t, repeats=None):
    p = pred if repeats is None else pred[pred["repeat"].isin(repeats)]
    e = np.abs(p[f"pred_{t}"].to_numpy() - truth.loc[p["key"], t].to_numpy())
    return pd.DataFrame({"pid": p["patient_id"].to_numpy(), "repeat": p["repeat"].to_numpy(), "e": e}).groupby(["pid", "repeat"])["e"].mean().unstack()


def regression_table(models):
    rows, const = [], models["constant"]
    for name, pred in models.items():
        reps = sorted(pred["repeat"].unique())
        for t in REG_TARGETS:
            if pred[f"pred_{t}"].isna().all():
                continue
            E = person_errors(pred, t)
            e = E.mean(axis=1).to_numpy()
            ec = person_errors(const, t, reps).loc[E.index].mean(axis=1).to_numpy()
            idx = RNG.integers(0, len(e), size=(B, len(e)))
            mb, cb = e[idx].mean(1), ec[idx].mean(1)
            p0 = pred[pred["repeat"] == reps[0]]
            yh, y = p0[f"pred_{t}"].to_numpy(), truth.loc[p0["key"], t].to_numpy()
            rows.append({"model": name, "target": t, "repeats": len(reps), "mae": e.mean(),
                         "mae_lo": np.percentile(mb, 2.5), "mae_hi": np.percentile(mb, 97.5), "mae_constant": ec.mean(),
                         "skill": 1 - e.mean() / ec.mean(), "skill_lo": np.percentile(1 - mb / cb, 2.5),
                         "skill_hi": np.percentile(1 - mb / cb, 97.5), "rmse": float(np.sqrt(np.mean((yh - y) ** 2))),
                         "r2": float(1 - ((yh - y) ** 2).sum() / ((y - y.mean()) ** 2).sum()),
                         "pearson_r": float(pearsonr(yh, y)[0]) if np.std(yh) > 1e-9 else np.nan,
                         "calibration_slope": float(np.polyfit(yh, y, 1)[0]) if np.std(yh) > 1e-9 else np.nan})
    return pd.DataFrame(rows)


def classification_table(models):
    rows = []
    for name, pred in models.items():
        if pred["pred_sex"].isna().all():
            continue
        per = []
        for _, p in pred.groupby("repeat"):
            y = truth.loc[p["key"], "sex_label"].to_numpy()
            ok = np.isfinite(y)
            prob, y = p["pred_sex"].to_numpy()[ok], y[ok].astype(int)
            per.append({"accuracy": np.mean((prob >= 0.5) == y), "balanced_accuracy": balanced_accuracy_score(y, (prob >= 0.5).astype(int)),
                        "auc": roc_auc_score(y, prob) if np.std(prob) > 0 else 0.5})
        rows.append({"model": name, "repeats": pred["repeat"].nunique(), **pd.DataFrame(per).mean().to_dict()})
    return pd.DataFrame(rows)


def compare(models, a, b):
    """Paired bootstrap of MAE(a) - MAE(b) on the shuffles both models share."""
    rows = []
    reps = sorted(set(models[a]["repeat"]) & set(models[b]["repeat"]))
    for t in REG_TARGETS:
        Ea = person_errors(models[a], t, reps).mean(axis=1)
        Eb = person_errors(models[b], t, reps).mean(axis=1).loc[Ea.index]
        d = (Ea - Eb).to_numpy()
        lo, hi = np.percentile(d[RNG.integers(0, len(d), size=(B, len(d)))].mean(1), [2.5, 97.5])
        rows.append({"A": a, "B": b, "target": NAME[t], "MAE A": Ea.mean(), "MAE B": Eb.mean(), "A − B": d.mean(),
                     "95% CI low": lo, "95% CI high": hi, "verdict": "A better" if hi < 0 else ("B better" if lo > 0 else "no clear difference")})
    return rows

In [ ]:
# ▶ Step 9.2 — Compute every metric (about a minute: 2,000 bootstrap resamples per model and target)
MODEL_NAMES = ["constant", "cnn", "video_specialists", "mtl_video_all", "stl_video_all", "physio_only",
               "multimodal_specialists", "mtl_video_physio", "state_constant", "demo_oracle"]
ROUTE_NAMES = [f"route_{r}" for r in list(VIDEO_ROUTES) + list(CHAIN_VIDEO) + list(PHYSIO_ROUTES) + list(CHAIN_MM)]
models = {n: m for n in MODEL_NAMES + ROUTE_NAMES if (m := load_oof(n)) is not None}
t0 = time.time()
reg_all = regression_table(models)                                    # every shuffle each model has
reg_r0 = regression_table({n: m[m["repeat"] == 0] for n, m in models.items()})   # shuffle 0: same people as the CNN
cls = classification_table(models)
reg_all.to_csv(MET / "regression_metrics.csv", index=False)
reg_r0.to_csv(MET / "regression_metrics_r0.csv", index=False)
cls.to_csv(MET / "classification_metrics.csv", index=False)
print(f"done in {time.time() - t0:.0f}s")
MAIN = ["constant", "cnn", "video_specialists", "mtl_video_all", "multimodal_specialists"]
tab = reg_r0[reg_r0.model.isin(MAIN)].assign(cell=lambda d: d.mae.map("{:.3g}".format) + " (" + d.skill.map(signed) + ")")
tab = tab.pivot(index="target", columns="model", values="cell")[MAIN].rename(index=NAME, columns=LABEL)
display(Markdown("**MAE in the target's unit (skill vs the median guess), shuffle 0: every model on the same people**"))
tab

In [ ]:
# ▶ Step 9.3 — Sex: how often the model is right
cls.assign(model=cls.model.map(lambda m: LABEL.get(m, m))).set_index("model").round(3)

In [ ]:
# ▶ Step 9.4 — Paired comparisons: is the difference real?
pairs = [("video_specialists", "cnn"), ("video_specialists", "constant"), ("multimodal_specialists", "video_specialists"),
         ("mtl_video_all", "video_specialists"), ("stl_video_all", "mtl_video_all")]
cmp = pd.DataFrame([row for a, b in pairs if a in models and b in models for row in compare(models, a, b)])
cmp.to_csv(MET / "comparisons.csv", index=False)
cmp.round(3)

**How to read it.** `A − B` below 0 means model A makes smaller errors. The verdict says *A better* or *B better* only
when the whole 95% interval is on one side of 0. Otherwise the difference could be luck.

## Step 10 · Result graphs

Each graph answers one question. Colours always mean the same model: grey = median guess, orange = your CNN,
blue = video specialists, aqua = shared network, violet = video + PPG + ECG, yellow = sensors only.

### 10.1 · Which targets can be predicted? (skill with 95% confidence intervals)

In [ ]:
# ▶ Step 10.1 — Graph: skill per target with confidence intervals
order = reg_r0[reg_r0.model == "video_specialists"].set_index("target")["skill"].sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(9, 5.2))
for off, m in zip((-0.22, 0, 0.22), ("cnn", "video_specialists", "multimodal_specialists")):
    d = reg_r0[reg_r0.model == m].set_index("target").reindex(order)
    y = np.arange(len(order)) + off
    ax.hlines(y, d["skill_lo"], d["skill_hi"], color=COLORS[m], lw=2)
    ax.scatter(d["skill"], y, s=46, color=COLORS[m], edgecolor=SURFACE, lw=1.5, zorder=3, label=LABEL[m])
ax.axvline(0, color=INK, lw=1)
ax.set_yticks(range(len(order)), [NAME[t] for t in order])
ax.set_xlabel("skill = 1 − MAE(model) / MAE(median guess)     (0 = no better than guessing)")
ax.legend(loc="lower right"); ax.set_title("Which targets can be predicted? (shuffle 0, 95% person-bootstrap intervals)")
fig.tight_layout(); savefig(fig, "10_skill_forest"); plt.show()

**What it means.** Age, BMI (and sex, Step 9.3) are clear wins from video. The physiological targets stay near 0 from
video; only arterial stiffness gains a little, from the pulse signals. Contact sensors move breathing rate and arterial
stiffness further to the right.

### 10.2 · Every model at a glance (skill, averaged over the 3 shuffles)

In [ ]:
# ▶ Step 10.2 — Graph: skill heatmap of all models
ms = [m for m in COLORS if m in set(reg_all.model)]
P = reg_all.pivot(index="model", columns="target", values="skill").reindex(ms)[REG_TARGETS]
lo = reg_all.pivot(index="model", columns="target", values="skill_lo").reindex(ms)[REG_TARGETS]
fig, ax = plt.subplots(figsize=(10.5, 0.5 * len(ms) + 1.6))
ax.imshow(P.clip(-0.5, 0.5).to_numpy(float), cmap=DIVERGING, vmin=-0.5, vmax=0.5, aspect="auto")
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        v = P.iat[i, j]
        if np.isfinite(v):
            sig = np.isfinite(lo.iat[i, j]) and lo.iat[i, j] > 0
            ax.text(j, i, signed(v) + ("*" if sig else ""), ha="center", va="center", fontsize=8,
                    color="white" if abs(v) > 0.33 else INK, fontweight="bold" if sig else "normal")
ax.set_xticks(range(len(REG_TARGETS)), [SHORT[t] for t in REG_TARGETS]); ax.set_yticks(range(len(ms)), [LABEL[m] for m in ms])
ax.grid(False); ax.set_title("Skill vs the median guess  (* = 95% interval entirely above 0)", fontsize=10)
fig.tight_layout(); savefig(fig, "11_skill_heatmap"); plt.show()

### 10.3 · Errors in real units

In [ ]:
# ▶ Step 10.3 — Graph: MAE per target with 95% intervals (line = median guess)
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, t in zip(axes.flat, REG_TARGETS):
    d = reg_r0[(reg_r0.target == t) & reg_r0.model.isin(MAIN)].set_index("model").reindex(MAIN).dropna(subset=["mae"])
    x = np.arange(len(d))
    ax.bar(x, d["mae"], width=0.6, color=[COLORS[m] for m in d.index])
    ax.errorbar(x, d["mae"], yerr=[d["mae"] - d["mae_lo"], d["mae_hi"] - d["mae"]], fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
    ax.axhline(d.loc["constant", "mae"], color=INK, lw=1)
    ax.set_xticks(x, [LABEL[m].split(": ")[-1] for m in d.index], rotation=35, ha="right")
    ax.set_ylim(d["mae_lo"].min() * 0.8, d["mae_hi"].max() * 1.05); ax.set_title(NAME[t]); ax.set_ylabel("MAE")
fig.suptitle("Average error per target (lower is better)", y=1.0); fig.tight_layout(); savefig(fig, "12_mae_bars"); plt.show()

### 10.4 · Predicted vs true

In [ ]:
# ▶ Step 10.4 — Graph: predicted vs true for the video specialists (black line = perfect)
p = load_oof("video_specialists", 0)
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for ax, t in zip(axes.flat, REG_TARGETS):
    y, yh, st = truth.loc[p["key"], t].to_numpy(), p[f"pred_{t}"].to_numpy(), truth.loc[p["key"], "step"].to_numpy()
    for s, c in (("before", COLORS["video_specialists"]), ("after", COLORS["cnn"])):
        ax.scatter(y[st == s], yh[st == s], s=14, color=c, alpha=0.6, edgecolor="none", label=f"{s} exercise")
    lo_, hi_ = np.percentile(np.r_[y, yh], [0.5, 99.5])
    ax.plot([lo_, hi_], [lo_, hi_], color=INK, lw=1); ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_)
    rr = f"{np.corrcoef(y, yh)[0, 1]:.2f}" if np.std(yh) > 1e-9 else "n/a (median guess)"
    ax.text(0.03, 0.95, f"r = {rr}\nMAE = {np.mean(np.abs(y - yh)):.3g}", transform=ax.transAxes, va="top", fontsize=8, color=INK2)
    ax.set_title(NAME[t]); ax.set_xlabel("true"); ax.set_ylabel("predicted")
axes[0, 0].legend(loc="lower right")
fig.suptitle("Flat clouds = the model stays near the median (it has little to go on)", y=1.0)
fig.tight_layout(); savefig(fig, "13_predicted_vs_true"); plt.show()

### 10.5 · Agreement (Bland–Altman), the standard clinical check

In [ ]:
# ▶ Step 10.5 — Graph: Bland–Altman plots for the multimodal model
p = load_oof("multimodal_specialists", 0)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, t in zip(axes.flat, ["saturation", "temperature", "respiratory", "rigidity", "glycated_hemoglobin", "cholesterol"]):
    y, yh = truth.loc[p["key"], t].to_numpy(), p[f"pred_{t}"].to_numpy()
    d = yh - y
    ax.scatter((y + yh) / 2, d, s=12, color=COLORS["multimodal_specialists"], alpha=0.5, edgecolor="none")
    for v, lab in ((d.mean(), "bias"), (d.mean() + 1.96 * d.std(), "+1.96 SD"), (d.mean() - 1.96 * d.std(), "−1.96 SD")):
        ax.axhline(v, color=INK if lab == "bias" else INK2, lw=1)
        ax.text(1.0, v, f" {lab} {signed(v)}", transform=ax.get_yaxis_transform(), va="center", fontsize=7, color=INK2)
    ax.set_title(NAME[t]); ax.set_xlabel("mean of predicted and true"); ax.set_ylabel("predicted − true")
fig.suptitle("95% of errors fall between the ±1.96 SD lines; a sloping cloud means pull towards the middle", y=1.0)
fig.tight_layout(); savefig(fig, "14_bland_altman"); plt.show()

### 10.6 · Sex classification

In [ ]:
# ▶ Step 10.6 — Graph: ROC curves and the confusion matrix
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.3, 1]})
for m in ["cnn", "video_specialists", "multimodal_specialists"]:
    p = load_oof(m, 0)
    y = truth.loc[p["key"], "sex_label"].to_numpy()
    ok = np.isfinite(y)
    fpr, tpr, _ = roc_curve(y[ok], p["pred_sex"].to_numpy()[ok])
    a1.plot(fpr, tpr, color=COLORS[m], label=f"{LABEL[m]} (AUC {auc_area(fpr, tpr):.3f})")
a1.plot([0, 1], [0, 1], color=INK2, lw=1); a1.legend(loc="lower right")
a1.set_xlabel("false positive rate (woman called man)"); a1.set_ylabel("true positive rate (man called man)")
a1.set_title("ROC curves (shuffle 0)")
p = load_oof("video_specialists", 0)
y = truth.loc[p["key"], "sex_label"].to_numpy()
ok = np.isfinite(y)
cm = confusion_matrix(y[ok].astype(int), (p["pred_sex"].to_numpy()[ok] >= 0.5).astype(int))
a2.imshow(cm, cmap=LinearSegmentedColormap.from_list("b", ["#f0efec", COLORS["video_specialists"]]))
for i in range(2):
    for j in range(2):
        a2.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, fontweight="bold", color="white" if cm[i, j] > cm.max() / 2 else INK)
a2.set_xticks([0, 1], ["predicted F", "predicted M"]); a2.set_yticks([0, 1], ["true F", "true M"]); a2.grid(False)
a2.set_title("Video specialists: sessions right and wrong")
fig.tight_layout(); savefig(fig, "15_sex"); plt.show()

### 10.7 · Where does the age error come from?

In [ ]:
# ▶ Step 10.7 — Graph: age and BMI error by age group
band = pd.cut(truth["age"], [0, 24, 30, 200], labels=["18-24", "25-30", ">30"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (t, unit) in zip(axes, (("age", "years"), ("bmi", "kg/m²"))):
    for i, m in enumerate(["constant", "cnn", "video_specialists", "multimodal_specialists"]):
        p = load_oof(m, 0)
        e = pd.DataFrame({"e": np.abs(p[f"pred_{t}"].to_numpy() - truth.loc[p["key"], t].to_numpy()),
                          "pid": p["patient_id"].to_numpy(), "band": band.loc[p["key"]].to_numpy()})
        v = e.groupby(["band", "pid"], observed=True)["e"].mean().groupby("band", observed=True).mean().reindex(["18-24", "25-30", ">30"])
        ax.bar(np.arange(3) + (i - 1.5) * 0.2, v.to_numpy(), width=0.18, color=COLORS[m], label=LABEL[m])
    n = people.assign(b=pd.cut(people["age"], [0, 24, 30, 200], labels=["18-24", "25-30", ">30"]))["b"].value_counts().reindex(["18-24", "25-30", ">30"])
    ax.set_xticks(range(3), [f"{b}\n({k} people)" for b, k in n.items()]); ax.set_ylabel(f"MAE ({unit})")
    ax.set_title(f"{NAME[t]}: error by age group")
axes[0].legend(loc="upper left")
fig.tight_layout(); savefig(fig, "16_error_by_age_group"); plt.show()

**What it means.** For the 545 people aged 18–24, guessing 20 is already within about 1.2 years, and the model adds
little. Its gain comes from the 55 older people, and only 45 of them are over 30, so be careful claiming anything for
older populations.

### 10.8 · Which input carries the information?

In [ ]:
# ▶ Step 10.8 — Graph: skill of every feature route (each tuned by its own inner loop)
routes = ["route_dino", "route_clip", "route_appearance", "route_temporal", "route_video_all", "route_video_all+demo",
          "route_demo", "route_ppg", "route_ecg", "route_physio", "route_video+physio", "route_video+physio+demo"]
rnames = {"route_dino": "DINOv2 face", "route_clip": "CLIP face", "route_appearance": "DINOv2 + CLIP",
          "route_temporal": "rPPG pulse signals", "route_video_all": "all video", "route_video_all+demo": "all video + predicted age/sex/BMI",
          "route_demo": "predicted age/sex/BMI only", "route_ppg": "finger PPG", "route_ecg": "ECG", "route_physio": "PPG + ECG",
          "route_video+physio": "video + PPG + ECG", "route_video+physio+demo": "video + PPG + ECG + predicted demo"}
routes = [r for r in routes if r in set(reg_all.model)]
P = reg_all.pivot(index="model", columns="target", values="skill").reindex(routes)[REG_TARGETS]
lo = reg_all.pivot(index="model", columns="target", values="skill_lo").reindex(routes)[REG_TARGETS]
fig, ax = plt.subplots(figsize=(10.5, 0.45 * len(routes) + 1.6))
ax.imshow(P.clip(-0.5, 0.5).to_numpy(float), cmap=DIVERGING, vmin=-0.5, vmax=0.5, aspect="auto")
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        v, sig = P.iat[i, j], lo.iat[i, j] > 0
        ax.text(j, i, signed(v) + ("*" if sig else ""), ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.33 else INK, fontweight="bold" if sig else "normal")
ax.axhline(6.5, color=INK, lw=1)
ax.set_xticks(range(len(REG_TARGETS)), [SHORT[t] for t in REG_TARGETS]); ax.set_yticks(range(len(routes)), [rnames[r] for r in routes])
ax.grid(False); ax.set_title("Above the line: camera only (deployable). Below: needs contact sensors.", fontsize=10)
fig.tight_layout(); savefig(fig, "17_modality_ablation"); plt.show()

### 10.9 · Shared network vs separate models

In [ ]:
# ▶ Step 10.9 — Graph: specialists vs one shared network vs the same network trained per target (shuffle 0)
ms3 = ["video_specialists", "mtl_video_all", "stl_video_all"]
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(REG_TARGETS))
for i, m in enumerate(ms3):
    d = reg_r0[reg_r0.model == m].set_index("target").reindex(REG_TARGETS)
    ax.bar(x + (i - 1) * 0.26, d["skill"], width=0.24, color=COLORS[m], label=LABEL[m])
ax.axhline(0, color=INK, lw=1); ax.set_xticks(x, [SHORT[t] for t in REG_TARGETS]); ax.set_ylabel("skill vs median guess")
ax.legend(loc="upper left"); ax.set_title("Shared network: better for BMI, worse for age (negative transfer); it cannot abstain (temperature)")
fig.tight_layout(); savefig(fig, "18_shared_vs_separate"); plt.show()

### 10.10 · What did the inner loop choose?

In [ ]:
# ▶ Step 10.10 — Graph: chosen model family and feature route per target (video specialists, 15 outer folds)
sel = pd.concat([pd.read_csv(f) for f in sorted(OOF.glob("selection_video_specialists_r*.csv"))])
sel["family"] = sel["config"].str.extract(r"^([a-z]+)", expand=False)
sel.loc[sel["family"] == "const", "route"] = "none (median guess)"     # the route does not matter when abstaining
palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, col in zip(axes, ("family", "route")):
    tb = sel.groupby(["target", col]).size().unstack(fill_value=0)
    tb = tb.div(tb.sum(axis=1), axis=0).reindex([t for t in ALL_TARGETS if t in tb.index])
    left = np.zeros(len(tb))
    for i, c in enumerate(tb.columns):
        ax.barh(range(len(tb)), tb[c], left=left, color=palette[i % 8], edgecolor=SURFACE, height=0.6, label=c)
        left += tb[c].to_numpy()
    ax.set_yticks(range(len(tb)), [SHORT[t] for t in tb.index]); ax.invert_yaxis(); ax.grid(False)
    ax.set_xlabel("share of the 15 outer folds"); ax.set_title(f"Chosen {'model family' if col == 'family' else 'feature route'}")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=4)
fig.tight_layout(); savefig(fig, "19_inner_loop_choices"); plt.show()

### 10.11 · Continuity with the earlier Face_AI results (the same 90 test people)

In [ ]:
# ▶ Step 10.11 — Table + graph: the 90 people of the old 420/90/90 test split, old and new models
split = pd.read_csv(OLD / "participant_split.csv", dtype={"patient_id": str}) if (OLD / "participant_split.csv").exists() else None
rows = {}
if split is not None:
    test_people = set(split.loc[split["split"] == "test", "patient_id"])
    sources = {"old CNN (Face_AI baseline)": OLD / "baseline_test" / "results" / "test_predictions.csv",
               "improved CNN notebook (Face_AI, 16 Oct)": OLD / "experiments" / "improved_cnn" / "results" / "test_predictions.csv"}
    preds = {k: pd.read_csv(v, dtype={"patient_id": str}) for k, v in sources.items() if v.exists()}
    preds.update({"your CNN (this CV)": load_oof("cnn", 0), "video specialists (this CV)": load_oof("video_specialists", 0),
                  "multimodal (this CV)": load_oof("multimodal_specialists", 0)})
    for label, p in preds.items():
        p = p[p["patient_id"].isin(test_people)]
        rows[label] = {NAME[t]: person_mae(np.abs(p[f"pred_{t}"].to_numpy() - truth.loc[p["key"], t].to_numpy()), p["patient_id"])
                       for t in ["age", "bmi", "respiratory", "rigidity"]}
        sc = "pred_prob_male" if "pred_prob_male" in p else "pred_sex"
        y = truth.loc[p["key"], "sex_label"].to_numpy()
        ok = np.isfinite(y)
        rows[label]["sex accuracy"] = np.mean((p[sc].to_numpy()[ok] >= 0.5) == y[ok])
hist90 = pd.DataFrame(rows).T
hist90.to_csv(MET / "historical_test_comparison.csv")
display(Markdown("Old models were trained on 420 people, new ones on about 480. Read this as continuity, not a controlled test."))
hist90.round(3)

### 10.12 · Head-to-head with your newest Face_AI model (same 510 training people, same 90 test people)

**What.** Step 10.11 compares models that learned from different people. Here the specialists are **refitted on exactly
the 510 people your newest model learned from** (your train + validation groups). Then both models are scored on your
90 test people. The inner loop chooses every model from those 510 people only, exactly as in Step 7.

**Why.** Both models now have the same training people, the same test people, the same labels and the same score (MAE
per person; the median guess comes from the 510). The only difference left is the method. The 95% intervals are
**paired**: every bootstrap resample uses the same people for both models.

In [ ]:
# ▶ Step 10.12a — Refit the specialists on your 510 train + validation people, predict your 90 test people (about 1 min)
FIXED_FILES = {tier: OOF / f"fixed_split_{tier}.csv" for tier in ("video_specialists", "multimodal_specialists")}


def run_fixed_split(role):
    """Step 7.4 for one fixed split: outer fold 0 = your test people, fold 1 = the people the models may learn from."""
    fold = np.where(role == "test", 0, 1)
    route_oof, inner_store, logs = {}, {}, []
    for route, blist in {**VIDEO_ROUTES, **PHYSIO_ROUTES}.items():
        route_oof[route], log, inner_store[route] = run_route(route, blist, fold, 0, outer=[0])
        logs.append(log)
    stage1 = select_tier(pd.concat(logs, ignore_index=True), list(VIDEO_ROUTES), 0)
    D = np.zeros((len(df), 3))                                   # stacking: predicted age, BMI, sex (as in Step 7.4)
    for c, t in enumerate(["age", "bmi", "sex"]):
        rt = stage1[stage1["target"] == t]["route"].iloc[0]
        D[fold == 1, c] = inner_store[rt][(0, t)]                # honest inner predictions for the training people
        D[fold == 0, c] = route_oof[rt][fold == 0, ALL_TARGETS.index(t)]
    for route, blist in {**CHAIN_VIDEO, **CHAIN_MM}.items():
        route_oof[route], log, _ = run_route(route, blist, fold, 0, extra={0: D}, outer=[0])
        logs.append(log)
    L = pd.concat(logs, ignore_index=True)
    for tier, path in FIXED_FILES.items():
        best = select_tier(L, TIERS[tier], 0)
        P = assemble(best, route_oof, fold)[fold == 0]
        out = df.loc[fold == 0, ["key", "patient_id", "step"]].copy()
        for j, t in enumerate(ALL_TARGETS):
            out[f"pred_{t}"] = P[:, j]
        out.to_csv(path, index=False)
        best.assign(tier=tier).to_csv(OOF / f"fixed_split_choice_{tier}.csv", index=False)


fixed_role = df["patient_id"].map(split.set_index("patient_id")["split"]).to_numpy() if split is not None else None
if fixed_role is not None and (RECOMPUTE_SPECIALISTS or not all(f.exists() for f in FIXED_FILES.values())):
    t0 = time.time()
    run_fixed_split(fixed_role)
    print(f"refitted on {df.loc[fixed_role != 'test', 'patient_id'].nunique()} people in {time.time() - t0:.0f}s")
print("fixed-split predictions:", "ready" if all(f.exists() for f in FIXED_FILES.values()) else "not available (no participant_split.csv)")

In [ ]:
# ▶ Step 10.12b — Table: your newest model vs the specialists on your 90 test people
THEIR_FILE = OLD / "experiments" / "improved_cnn" / "results" / "cv" / "step13_test_predictions.csv"   # ← your newest test predictions


def head_to_head(their_file):
    """Per-person MAE of your model and of both specialist tiers, with paired bootstrap intervals of the difference."""
    yours = pd.read_csv(their_file, dtype={"patient_id": str}).set_index("key")
    keys = sorted(yours.index)
    mine = {tier: pd.read_csv(f, dtype={"patient_id": str}).set_index("key") for tier, f in FIXED_FILES.items()}
    assert all(set(m.index) == set(keys) for m in mine.values()), "the two files hold different test people"
    pid = truth.loc[keys, "patient_id"].to_numpy()
    n = len(np.unique(pid))
    boot = np.random.default_rng(SEED).integers(0, n, size=(B, n))           # the same resampled people for every model
    learn = fixed_role != "test"

    def per_person(pred, t):
        e = np.abs(np.asarray(pred, float) - truth.loc[keys, t].to_numpy())
        return pd.Series(e).groupby(pid).mean().to_numpy()                   # one error per person, sorted by person id

    rows = []
    for t in REG_TARGETS:
        e0 = per_person(np.full(len(keys), df.loc[learn, t].median()), t)
        ey = per_person(yours.loc[keys, f"pred_{t}"], t)
        for tier in FIXED_FILES:
            em = per_person(mine[tier].loc[keys, f"pred_{t}"], t)
            d = em[boot].mean(1) - ey[boot].mean(1)
            rows.append({"target": t, "model": tier, "guess": e0.mean(), "yours": ey.mean(), "mine": em.mean(),
                         "delta": em.mean() - ey.mean(), "lo": np.percentile(d, 2.5), "hi": np.percentile(d, 97.5)})
    y = truth.loc[keys, "sex_label"].to_numpy()
    ok = np.isfinite(y)
    sex = {}
    for name, prob in [("your newest model", yours.loc[keys, "pred_prob_male"])] + [
            (LABEL[tier], mine[tier].loc[keys, "pred_sex"]) for tier in FIXED_FILES]:
        prob = np.asarray(prob, float)[ok]
        hat = (prob >= 0.5).astype(int)
        sex[name] = {"accuracy": np.mean(hat == y[ok]), "balanced accuracy": balanced_accuracy_score(y[ok].astype(int), hat),
                     "AUC": roc_auc_score(y[ok], prob), "wrong recordings": f"{int((hat != y[ok]).sum())} of {int(ok.sum())}"}
    band = np.where(truth.loc[keys, "age"].to_numpy() <= 24, "18-24", "older than 24")
    ages = pd.DataFrame({"people": pd.Series(pid).groupby(band).nunique()})
    for name, pred in [("your newest model", yours.loc[keys, "pred_age"]), (LABEL["video_specialists"], mine["video_specialists"].loc[keys, "pred_age"])]:
        e = pd.DataFrame({"e": np.abs(np.asarray(pred, float) - truth.loc[keys, "age"].to_numpy()), "pid": pid, "band": band})
        ages[f"age MAE: {name}"] = e.groupby(["band", "pid"])["e"].mean().groupby("band").mean()
    return pd.DataFrame(rows), pd.DataFrame(sex).T, ages


h2h = None
if fixed_role is not None and THEIR_FILE.exists() and all(f.exists() for f in FIXED_FILES.values()):
    h2h, h2h_sex, h2h_age = head_to_head(THEIR_FILE)
    h2h.to_csv(MET / "head_to_head_fixed_split.csv", index=False)
    verdict = lambda r: "specialists better" if r["hi"] < 0 else ("yours better" if r["lo"] > 0 else "no clear difference")
    ci = lambda r: f"{signed(r['delta'], 3)} [{signed(r['lo'], 3)}, {signed(r['hi'], 3)}]"
    show = {}
    for t in REG_TARGETS:
        v, m = (h2h[(h2h.target == t) & (h2h.model == tier)].iloc[0] for tier in FIXED_FILES)
        show[NAME[t]] = {"median guess": f"{v['guess']:#.3g}", "your newest model": f"{v['yours']:#.3g}",
                         "video specialists": f"{v['mine']:#.3g}", "video − yours [95% CI]": ci(v), "verdict (video only)": verdict(v),
                         "video + PPG + ECG": f"{m['mine']:#.3g}", "multimodal − yours [95% CI]": ci(m)}
    display(Markdown(f"**MAE on your test people** (`{THEIR_FILE.name}`). Below 0 = the specialists make smaller errors."))
    display(pd.DataFrame(show).T)
    display(Markdown("**Sex** (every recording)"), h2h_sex.round(3))
    display(Markdown("**Where the age error comes from**"), h2h_age.round(2))
else:
    print("Skipped: needs participant_split.csv and your newest test predictions from the Face_AI folder.")

In [ ]:
# ▶ Step 10.12c — Graph: who wins where (difference in MAE as % of the median guess's MAE)
if h2h is not None:
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    tg = REG_TARGETS[::-1]
    for off, tier in zip((0.17, -0.17), FIXED_FILES):
        g = h2h[h2h.model == tier].set_index("target").reindex(tg)
        yy = np.arange(len(tg)) + off
        ax.hlines(yy, 100 * g["lo"] / g["guess"], 100 * g["hi"] / g["guess"], color=COLORS[tier], lw=2)
        ax.scatter(100 * g["delta"] / g["guess"], yy, s=46, color=COLORS[tier], edgecolor=SURFACE, lw=1.5, zorder=3, label=LABEL[tier])
    ax.axvline(0, color=INK, lw=1)
    ax.set_ylim(-1.1, len(tg) - 0.5)
    ax.text(-0.6, -0.8, "← specialists better", ha="right", va="center", fontsize=9, color=INK2)
    ax.text(0.6, -0.8, "your model better →", ha="left", va="center", fontsize=9, color=INK2)
    ax.set_yticks(range(len(tg)), [NAME[t] for t in tg])
    ax.set_xlabel("MAE(specialists) − MAE(your newest model), in % of the median guess's MAE")
    ax.legend(loc="upper left"); ax.set_title("Same 510 training people, same 90 test people (paired 95% intervals)")
    fig.tight_layout(); savefig(fig, "22_head_to_head"); plt.show()

**What it means** (run of 19 October 2026, against your Step 13 predictions):

* **BMI:** the specialists are clearly better (about 1.97 vs 2.45 kg/m²). Their BMI model uses the **CLIP** features,
  which your model does not have.
* **Arterial stiffness and stress:** the specialists are better. Stiffness comes from boosted trees on the skin-colour
  pulse features. For stress, the specialists stay close to the median guess, while your model does worse than it.
* **Age:** your model is better (about 1.50 vs 1.70 years), and the interval just touches 0. Its gain is among the
  people aged 18–24, where your MLP heads, trained on augmented crops, separate small age differences better than a
  linear ridge.
* **SpO2, temperature, HbA1c, cholesterol and, from video alone, breathing rate:** no clear difference. Nobody beats the
  median guess here.
* **With PPG + ECG**, breathing rate and stiffness improve a lot. That column needs contact sensors, so it is not a
  video model.
* **Sex:** both models make only a few mistakes, a tie in practice.

**Be careful:** 90 people give wide intervals. Neither side is a pristine test either: your versions were compared on
these 90 people, and the specialists were designed with cross-validation on all 600 people, which include them.

## Step 11 · Overfitting curves

**What.** Two classic diagnostics, using the face features and ridge regression on shuffle 0:

* **Validation curve.** Training error and new-people error as the penalty grows.
  * Weak penalty: training error is tiny but new-people error is large. That is **overfitting**: the model memorises.
  * Huge penalty: both errors rise back to the median guess. That is **underfitting**: the model ignores the face.
* **Learning curve.** New-people error as we give the model more training *people*. If it is still falling at 480,
  more participants would help.

**Why.** They show *why* the penalties chosen in Step 7 matter, and whether we are limited by model or by data.

In [ ]:
# ▶ Step 11.1 — Compute the validation and learning curves (or load them)
VC_FILE, LC_FILE = MET / "validation_curve.csv", MET / "learning_curve.csv"


def ridge_one(Ztr, y, Zs, alpha):
    zm, ym = Ztr.mean(0), y.mean()
    U, s, Vt = np.linalg.svd(Ztr - zm, full_matrices=False)
    w = Vt.T @ ((s / (s ** 2 + alpha * len(Ztr))) * (U.T @ (y - ym)))
    return [(Z - zm) @ w + ym for Z in Zs]


if RECOMPUTE_DIAGNOSTICS or not (VC_FILE.exists() and LC_FILE.exists()):
    X, sizes, fold = np.column_stack([X_dino, X_clip]), [384, 768], session_folds(0)
    vrows, lrows = [], []
    for k in range(N_FOLDS):
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
        sc = BlockScaler(sizes).fit(X[tr])
        Ztr, Zte = sc.transform(X[tr]), sc.transform(X[te])
        for t, use_log in (("age", True), ("bmi", True), ("glycated_hemoglobin", False)):
            y = df[t].to_numpy(float)
            yf = np.log(y) if use_log else y
            for a in np.logspace(-5, 2, 15):
                ptr, pte = ridge_one(Ztr, yf[tr], [Ztr, Zte], a)
                ptr, pte = (np.exp(ptr), np.exp(pte)) if use_log else (ptr, pte)
                vrows.append({"fold": k, "target": t, "alpha": a, "train_mae": person_mae(np.abs(ptr - y[tr]), pids[tr]),
                              "test_mae": person_mae(np.abs(pte - y[te]), pids[te])})
        rng = np.random.default_rng(k)
        for n_people in (30, 60, 120, 240, 360, len(np.unique(pids[tr]))):
            for rep in range(3 if n_people < len(np.unique(pids[tr])) else 1):
                sub = tr[np.isin(pids[tr], rng.choice(np.unique(pids[tr]), n_people, replace=False))]
                sc2 = BlockScaler(sizes).fit(X[sub])
                Zs, Zt = sc2.transform(X[sub]), sc2.transform(X[te])
                for t in ("age", "bmi"):
                    y = df[t].to_numpy(float)
                    ptr, pte = ridge_one(Zs, np.log(y[sub]), [Zs, Zt], 1e-2)
                    lrows.append({"fold": k, "target": t, "n_people": n_people, "train_mae": person_mae(np.abs(np.exp(ptr) - y[sub]), pids[sub]),
                                  "test_mae": person_mae(np.abs(np.exp(pte) - y[te]), pids[te]),
                                  "const_mae": person_mae(np.abs(np.median(y[sub]) - y[te]), pids[te])})
    pd.DataFrame(vrows).to_csv(VC_FILE, index=False)
    pd.DataFrame(lrows).to_csv(LC_FILE, index=False)
vc, lc = pd.read_csv(VC_FILE), pd.read_csv(LC_FILE)
print("validation curve rows:", len(vc), "| learning curve rows:", len(lc))

In [ ]:
# ▶ Step 11.2 — Graph: validation curves (the gap between the lines is overfitting)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, t in zip(axes, ["age", "bmi", "glycated_hemoglobin"]):
    d = vc[vc.target == t].groupby("alpha")[["train_mae", "test_mae"]].mean()
    ax.plot(d.index, d["train_mae"], color=COLORS["mtl_video_all"], label="training people")
    ax.plot(d.index, d["test_mae"], color=COLORS["video_specialists"], label="new people")
    b = d["test_mae"].idxmin()
    ax.scatter([b], [d.loc[b, "test_mae"]], s=50, color=COLORS["video_specialists"], edgecolor=SURFACE, zorder=3)
    ax.set_xscale("log"); ax.set_xlabel("ridge penalty α (× sessions)  →  stronger"); ax.set_ylabel("MAE"); ax.set_title(NAME[t])
    ax.text(0.03, 0.04, "overfit", transform=ax.transAxes, color=INK2, fontsize=8)
    ax.text(0.97, 0.04, "underfit", transform=ax.transAxes, color=INK2, fontsize=8, ha="right")
axes[0].legend(loc="upper center")
fig.suptitle("Validation curves (face features + ridge). For HbA1c the curve is flat: the face adds almost nothing.", y=1.03)
fig.tight_layout(); savefig(fig, "20_validation_curves"); plt.show()

In [ ]:
# ▶ Step 11.3 — Graph: learning curves (does more data help?)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, t in zip(axes, ("age", "bmi")):
    d = lc[lc.target == t].groupby("n_people")[["train_mae", "test_mae", "const_mae"]].mean()
    ax.plot(d.index, d["train_mae"], marker="o", ms=5, color=COLORS["mtl_video_all"], label="training people")
    ax.plot(d.index, d["test_mae"], marker="o", ms=5, color=COLORS["video_specialists"], label="new people")
    ax.plot(d.index, d["const_mae"], color=COLORS["constant"], label="median guess")
    ax.set_xlabel("number of training PEOPLE"); ax.set_ylabel("MAE"); ax.set_title(NAME[t])
axes[0].legend()
fig.suptitle("Learning curves: the new-people error is flattening, so more people would help only a little", y=1.03)
fig.tight_layout(); savefig(fig, "21_learning_curves"); plt.show()

## Step 12 · Summary

In [ ]:
# ▶ Step 12 — The key numbers, written out from this notebook's results
R = reg_r0.set_index(["model", "target"])
sx = cls.set_index("model")
lines = [
    f"* **Age:** median guess {R.loc[('constant', 'age'), 'mae']:.2f} y → your CNN {R.loc[('cnn', 'age'), 'mae']:.2f} y → "
    f"video specialists **{R.loc[('video_specialists', 'age'), 'mae']:.2f} y** (r = {R.loc[('video_specialists', 'age'), 'pearson_r']:.2f}).",
    f"* **BMI:** {R.loc[('constant', 'bmi'), 'mae']:.2f} → {R.loc[('cnn', 'bmi'), 'mae']:.2f} → **{R.loc[('video_specialists', 'bmi'), 'mae']:.2f}** kg/m².",
    f"* **Sex:** accuracy {100 * sx.loc['cnn', 'accuracy']:.1f}% (CNN) → **{100 * sx.loc['video_specialists', 'accuracy']:.1f}%** (specialists).",
    f"* **Physiology from video:** skill between {R.xs('video_specialists').loc[['saturation', 'temperature', 'glycated_hemoglobin', 'cholesterol', 'respiratory', 'rigidity', 'stress'], 'skill'].min():+.2f} "
    f"and {R.xs('video_specialists').loc[['saturation', 'temperature', 'glycated_hemoglobin', 'cholesterol', 'respiratory', 'rigidity', 'stress'], 'skill'].max():+.2f}: at or near the median guess.",
    f"* **Adding PPG + ECG:** breathing rate skill {R.loc[('multimodal_specialists', 'respiratory'), 'skill']:+.2f}, "
    f"arterial stiffness {R.loc[('multimodal_specialists', 'rigidity'), 'skill']:+.2f} (stiffness is an upper bound: same kind of sensor).",
    f"* **Splitting by video** would have faked cholesterol skill {leak.iloc[0, 0]:+.2f}; split by person it is {leak.iloc[0, 1]:+.2f}.",
]
display(Markdown("\n".join(lines)))

**Next steps** (see `PLAN.md`):

1. Use all three cameras and the full three-minute recordings.
2. Try stronger face encoders, fine-tuned on the GPU, and add a single-task network for age as a candidate.
3. Train a camera pulse model with the finger PPG as its target.
4. Test how much survives on de-identified video (the project's privacy theme).

## Step 13 · Optional: update the presentation

**What.** Rewrites `presentation/data.json` from this notebook's results and rebuilds `presentation/index.html`
(double-click to open) and `presentation/artifact.html`. Set `UPDATE_PRESENTATION = True` in Step 0.1 to run it.

In [ ]:
# ▶ Step 13 — Rebuild the slides with this notebook's numbers
if UPDATE_PRESENTATION:
    r3 = lambda v: None if v is None or not np.isfinite(v) else round(float(v), 3)
    pres = PROJECT / "presentation"
    tmpl_cols = [ppg_names.index(f"ppg_tmpl_{j:02d}") for j in range(25)]
    dist = {}
    for t, step in (("saturation", 1.0), ("temperature", 0.1), ("respiratory", 1.0)):
        vcount = (df[t] / step).round().astype(int).value_counts().sort_index()
        dist[t] = {"values": [r3(v * step) for v in vcount.index], "counts": vcount.astype(int).tolist(), "step": step}
    w = df.pivot(index="patient_id", columns="step", values=["saturation", "temperature", "respiratory", "rigidity"])
    retest = {t: r3(spearmanr(w[(t, "before")], w[(t, "after")])[0]) for t in ["saturation", "temperature", "respiratory", "rigidity"]}
    retest["temperature_identical"] = r3(np.mean(w[("temperature", "before")] == w[("temperature", "after")]))
    MODELS_P = ["constant", "cnn", "video_specialists", "mtl_video_all", "stl_video_all", "physio_only", "multimodal_specialists", "state_constant"]
    sel_v = sel.groupby(["target", "family"]).size().unstack(fill_value=0)
    p_s = {m: load_oof(m, 0).set_index("key") for m in ["constant", "cnn", "video_specialists"]}
    keys = p_s["video_specialists"].index
    sub = reg_r0
    data = {
        "pulse": [r3(v) for v in np.nanmean(X_ppg[:, tmpl_cols], axis=0)],
        "counts": {"people": 600, "sessions": len(df), "videos": 3600}, "nRppg": int(X_rppg.shape[1]),
        "labelStats": {t: {"mean": r3(df[t].mean()), "sd": r3(df[t].std()), "min": r3(df[t].min()), "max": r3(df[t].max()),
                           "median": r3(df[t].median())} for t in REG_TARGETS},
        "dist": dist, "retest": retest,
        "timing": {"cuff": [r3(v) for v in timing["Cuff pulse, right after squats"]],
                   "ppg": [r3(v) for v in timing["Heart rate during the video (finger PPG)"]],
                   "rr": [r3(df.respiratory[after == 0].mean()), r3(df.respiratory[after == 1].mean())]},
        "stateAUC": {k: r3(v) for k, v in auc.items()},
        "leak": {"leaky": {t: r3(leak.loc[NAME[t]].iloc[0]) for t in LEAK_T}, "grouped": {t: r3(leak.loc[NAME[t]].iloc[1]) for t in LEAK_T},
                 "twinInTrain": r3(twin)},
        "ages": people["age"].astype(float).round(1).tolist(),
        "results": {m: {r.target: {"mae": r3(r.mae), "skill": r3(r.skill), "lo": r3(r.skill_lo), "hi": r3(r.skill_hi), "r": r3(r.pearson_r)}
                        for r in sub[sub.model == m].itertuples()} for m in MODELS_P},
        "sex": {m: {"acc": r3(sx.loc[m, "accuracy"]), "bal": r3(sx.loc[m, "balanced_accuracy"]), "auc": r3(sx.loc[m, "auc"])}
                for m in MODELS_P if m in sx.index},
        "ablation": {rt: {r.target: {"skill": r3(r.skill), "lo": r3(r.skill_lo)} for r in reg_all[reg_all.model == rt].itertuples()}
                     for rt in ["route_dino", "route_clip", "route_appearance", "route_temporal", "route_video_all", "route_ppg",
                                "route_ecg", "route_physio", "route_video+physio"]},
        "valcurve": {t: {"alpha": [float(a) for a in g.index], "train": [r3(v) for v in g["train_mae"]], "test": [r3(v) for v in g["test_mae"]]}
                     for t, g in ((t, vc[vc.target == t].groupby("alpha")[["train_mae", "test_mae"]].mean()) for t in ["age", "bmi", "glycated_hemoglobin"])},
        "mtlcurve": (lambda c0: {"epoch": c0.epoch.tolist(), "train": [r3(v) for v in c0.train_loss], "val": [r3(v) for v in c0.val_loss]})(mtl_curves[mtl_curves.seed == mtl_curves.seed.min()]),
        "scatter": {t: {"true": [round(float(v), 2) for v in truth.loc[keys, t]],
                        **{m: [round(float(v), 2) for v in p_s[m].loc[keys, f"pred_{t}"]] for m in p_s}} for t in ["age", "bmi"]},
        "ageBands": {}, "selection": {t: {f: r3(sel_v.loc[t, f] / sel_v.loc[t].sum()) for f in sel_v.columns} for t in sel_v.index},
        "history": {"labels": ["Old CNN", "Improved CNN (Face_AI)", "New video model", "New multimodal"],
                    "age": [r3(hist90.iloc[i][NAME["age"]]) for i in (0, 1, 3, 4)] if len(hist90) >= 5 else [],
                    "bmi": [r3(hist90.iloc[i][NAME["bmi"]]) for i in (0, 1, 3, 4)] if len(hist90) >= 5 else []},
        "cnnParams": 533075,
    }
    bands_lbl = pd.cut(truth["age"], [0, 24, 30, 200], labels=["18-24", "25-30", ">30"])
    for lvl in ["18-24", "25-30", ">30"]:
        entry = {"n": int((pd.cut(people["age"], [0, 24, 30, 200], labels=["18-24", "25-30", ">30"]) == lvl).sum())}
        for m in ["constant", "cnn", "video_specialists"]:
            p = load_oof(m, 0)
            mk = (bands_lbl.loc[p["key"]] == lvl).to_numpy()
            entry[m] = r3(person_mae(np.abs(p.loc[mk, "pred_age"].to_numpy() - truth.loc[p.loc[mk, "key"], "age"].to_numpy()), p.loc[mk, "patient_id"]))
        data["ageBands"][lvl] = entry
    (pres / "data.json").write_text(json.dumps(data, separators=(",", ":")), encoding="utf-8")
    page = (pres / "template.html").read_text(encoding="utf-8").replace("__DECK_DATA__", json.dumps(data, separators=(",", ":")).replace("</", "<\\/"))
    head, body = page.split("<!--BODY-->", 1)
    (pres / "artifact.html").write_text(head + body, encoding="utf-8")
    (pres / "index.html").write_text("<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n"
                                     "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1, viewport-fit=cover\">\n"
                                     + head + "</head>\n<body>\n" + body + "</body>\n</html>\n", encoding="utf-8")
    print("presentation rebuilt:", pres / "index.html")
else:
    print("UPDATE_PRESENTATION is False: the slides were not changed.")